# Medication Feature Reduction for MIMIC-IV LOS Fairness Evaluation

This notebook implements the controlled **A1 medication-reduction experiment**. The completed No-Imputation reproduction is treated as the A0 baseline. A1 changes only the medication representation: raw drug-name indicators are replaced by standardized ATC level-3 (ATC3) pharmacological categories. The LOS definition, cohort, patient-level splits, non-medication predictors, XGBoost configuration, prediction threshold, and race/insurance evaluation are otherwise retained.

**Design rationale.** ATC3 was selected a priori as an intermediate pharmacological grouping that reduces sparse medication dimensionality without collapsing all medications into very broad anatomical/therapeutic groups. It was not selected by optimizing held-out model performance. NDC identifiers are used only as a fallback for medication names that cannot be resolved from text.


## 1. Environment and data paths

The analysis uses the completed No-Imputation cohort and the processed pharmacy checkpoint. Intermediate medication-mapping artifacts are stored separately under `medication_feature_reduction` so that the A0 reproduction remains unchanged.


In [ ]:
import os, json, time, random, gc
import numpy as np
import pandas as pd
import duckdb
import requests
from IPython.display import display

PROJECT_DIR = os.environ.get("MIMIC_FAIRNESS_WORKDIR")

if not PROJECT_DIR:
    raise RuntimeError(
        "Set MIMIC_FAIRNESS_WORKDIR to your private research workspace."
    )
DATA_DIR = os.path.join(PROJECT_DIR, "data")
PROCESSED_PHARM = os.path.join(DATA_DIR, "filtered_pharm_processed.parquet")
NOIMP_PARQUET = os.path.join(DATA_DIR, "icu_data_no_imputation_reproduction.parquet")
NOIMP_CSV = os.path.join(PROJECT_DIR, "icu_data_no_imputation_reproduction.csv")
REDUCTION_DIR = os.path.join(DATA_DIR, "medication_feature_reduction")
os.makedirs(REDUCTION_DIR, exist_ok=True)

DRUG_PREVALENCE_CSV = os.path.join(REDUCTION_DIR, "drug_prevalence_final_cohort.csv")
ATC_VERSION_JSON = os.path.join(REDUCTION_DIR, "atc_source_version.json")

if not os.path.exists(PROCESSED_PHARM):
    raise FileNotFoundError(PROCESSED_PHARM)
if not os.path.exists(NOIMP_PARQUET) and not os.path.exists(NOIMP_CSV):
    raise FileNotFoundError("Completed No-Imputation cohort not found.")

print("Project directory:", PROJECT_DIR)
print("Output directory:", REDUCTION_DIR)


## 2. Baseline cohort and raw medication sparsity

The medication audit is restricted to the same 74,501 ICU stays used by A0. Prevalence is computed for the final cohort only. The pharmacy checkpoint contains 5,222 raw medication columns, while 5,097 distinct medication names remain active in the final merged cohort.


In [ ]:
pharm_path = PROCESSED_PHARM.replace("\\\\", "/")
con = duckdb.connect()

if os.path.exists(NOIMP_PARQUET):
    cohort_path = NOIMP_PARQUET.replace("\\\\", "/")
    con.execute(f"""CREATE OR REPLACE TEMP VIEW final_cohort AS
        SELECT DISTINCT CAST(subject_id AS BIGINT) AS subject_id, CAST(stay_id AS BIGINT) AS stay_id
        FROM read_parquet('{cohort_path}')""")
else:
    cohort_path = NOIMP_CSV.replace("\\\\", "/")
    con.execute(f"""CREATE OR REPLACE TEMP VIEW final_cohort AS
        SELECT DISTINCT TRY_CAST(subject_id AS BIGINT) AS subject_id, TRY_CAST(stay_id AS BIGINT) AS stay_id
        FROM read_csv_auto('{cohort_path}', header=true, union_by_name=true)""")

cohort_n = con.execute("SELECT COUNT(*) FROM final_cohort").fetchone()[0]
checkpoint_drugs = con.execute(f"SELECT COUNT(DISTINCT drug) FROM read_parquet('{pharm_path}') WHERE drug IS NOT NULL").fetchone()[0]
assert cohort_n == 74501

drug_prevalence = con.execute(f"""
WITH final_pharm AS (
    SELECT DISTINCT CAST(p.subject_id AS BIGINT) AS subject_id, CAST(p.stay_id AS BIGINT) AS stay_id,
           TRIM(CAST(p.drug AS VARCHAR)) AS drug
    FROM read_parquet('{pharm_path}') p
    INNER JOIN final_cohort c USING(subject_id, stay_id)
    WHERE p.drug IS NOT NULL AND TRIM(CAST(p.drug AS VARCHAR)) <> ''
)
SELECT drug, COUNT(DISTINCT stay_id) AS stay_count,
       COUNT(DISTINCT stay_id) / {cohort_n}.0 AS prevalence
FROM final_pharm
GROUP BY drug
ORDER BY stay_count DESC, drug
""").fetchdf()
con.close()

drug_prevalence.to_csv(DRUG_PREVALENCE_CSV, index=False)
n_drugs = len(drug_prevalence)
pos = int(drug_prevalence['stay_count'].sum())
density = pos / (cohort_n * n_drugs)

print("Final cohort ICU stays:", cohort_n)
print("Medication columns in pharmacy checkpoint:", checkpoint_drugs)
print("Distinct medication names in final cohort:", n_drugs)
print("Positive stay-drug cells:", pos)
print(f"Raw medication density: {density:.6%}")
print(f"Raw medication sparsity: {1-density:.6%}")
display(drug_prevalence.head(20))


## 3. Standardized RxNorm and ATC mapping

Medication strings are resolved through RxNorm. Direct ATC properties are obtained from ingredient concepts rather than using broad RxClass relationship expansion. Valid multiple ATC assignments are retained. Precise ingredients (PIN) and multi-ingredient concepts (MIN) are subsequently resolved to constituent ingredient (IN) concepts when needed.


### 3.1 RxNav client and source provenance

The ATC source version is saved with the analysis, and transient HTTP failures are retried with exponential backoff.


In [ ]:
# Reproducible RxNav client with retry handling for transient service failures.
RXNAV_BASE = "https://rxnav.nlm.nih.gov/REST"
session = requests.Session()
session.headers.update({"User-Agent": "MIMIC-Fairness-ATC-Feature-Audit/1.0"})

def get_json(url, params=None, timeout=45, retries=5):
    """
    Robust RxNav GET request.

    Retries transient failures such as:
        - 429 rate limiting
        - 500/502/503/504 server failures
        - timeout / connection errors

    Client-side HTTP errors such as 400 are not repeatedly retried.
    """
    last_error = None
    for attempt in range(retries):
        try:
            response = session.get(url, params=params, timeout=timeout)
            if response.status_code in {429, 500, 502, 503, 504}:
                wait = 2 ** attempt + random.uniform(0.2, 1.0)
                print(f'Temporary HTTP {response.status_code}; retrying in {wait:.1f}s')
                time.sleep(wait)
                continue
            if 400 <= response.status_code < 500:
                raise RuntimeError(f'Non-retryable HTTP {response.status_code}\nURL: {response.url}\nResponse: {response.text[:500]}')
            response.raise_for_status()
            return response.json()
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
            last_error = e
            wait = 2 ** attempt + random.uniform(0.2, 1.0)
            print(f'Connection error; retrying in {wait:.1f}s')
            time.sleep(wait)
        except RuntimeError:
            raise
        except (requests.exceptions.HTTPError, ValueError) as e:
            last_error = e
            wait = 2 ** attempt + random.uniform(0.2, 1.0)
            time.sleep(wait)
    raise RuntimeError(f'Request failed after {retries} attempts.\nURL: {url}\nParameters: {params}\nLast error: {last_error}')

version_data = get_json(f"{RXNAV_BASE}/rxclass/version/ATC.json")
with open(ATC_VERSION_JSON, "w", encoding="utf-8") as f:
    json.dump(version_data, f, indent=2)
print("ATC source version:", version_data)


### 3.2 Text-to-ingredient and direct ATC mapping

The primary path resolves each medication name to RxNorm ingredient concepts and reads direct ATC properties from those concepts.


In [ ]:
# Text-based mapping: MIMIC drug string -> RxNorm ingredient concept -> direct ATC property.
def rxcui_by_name(drug_name):
    payload = get_json(f'{RXNAV_BASE}/rxcui.json', params={'name': drug_name, 'search': 2, 'allsrc': 0})
    ids = payload.get('idGroup', {}).get('rxnormId', []) or []
    return [str(x) for x in ids]

def direct_atc_codes_by_rxcui(rxcui):
    """
    Return ATC identifiers directly attached to an RxNorm concept.
    This avoids the broader RxClass default-path expansion.
    """
    payload = get_json(f'{RXNAV_BASE}/rxcui/{rxcui}/property.json', params={'propName': 'ATC'})
    concepts = payload.get('propConceptGroup', {}).get('propConcept', []) or []
    codes = sorted({str(x.get('propValue')).strip() for x in concepts if x.get('propValue')})
    return codes

def get_rxcui_properties(rxcui):
    """
    Get the canonical properties of an active RxNorm concept.
    """
    payload = get_json(f'{RXNAV_BASE}/rxcui/{rxcui}/properties.json')
    props = payload.get('properties', {}) or {}
    return {'rxcui': str(rxcui), 'rxnorm_name': props.get('name'), 'tty': props.get('tty')}

def get_related_ingredient_concepts(rxcui):
    """
    Retrieve ingredient-level concepts related to an RxNorm concept.

    IN  = Ingredient
    PIN = Precise Ingredient
    MIN = Multiple Ingredients
    """
    payload = get_json(f'{RXNAV_BASE}/rxcui/{rxcui}/related.json', params={'tty': 'IN PIN MIN'})
    groups = payload.get('relatedGroup', {}).get('conceptGroup', []) or []
    rows = []
    for group in groups:
        tty = group.get('tty')
        concepts = group.get('conceptProperties', []) or []
        for concept in concepts:
            rows.append({'ingredient_rxcui': str(concept.get('rxcui')), 'ingredient_name': concept.get('name'), 'ingredient_tty': tty})
    if rows:
        rows = pd.DataFrame(rows).drop_duplicates().to_dict('records')
    return rows

def resolve_drug_to_ingredients(drug_name):
    """
    Resolve an original MIMIC medication string to ingredient-level
    RxNorm concepts without following RxClass therapeutic relationships.
    """
    rxcuis = rxcui_by_name(drug_name)
    if not rxcuis:
        return ([], 'unmapped_name', [])
    ingredients = []
    for rxcui in rxcuis:
        props = get_rxcui_properties(rxcui)
        tty = props.get('tty')
        if tty in {'IN', 'PIN', 'MIN'}:
            ingredients.append({'ingredient_rxcui': str(rxcui), 'ingredient_name': props.get('rxnorm_name'), 'ingredient_tty': tty})
        else:
            ingredients.extend(get_related_ingredient_concepts(rxcui))
    if ingredients:
        ingredients = pd.DataFrame(ingredients).drop_duplicates().to_dict('records')
        return (ingredients, 'resolved', rxcuis)
    return ([], 'no_ingredient', rxcuis)

def map_drug_direct_atc(drug_name):
    ingredients, status, source_rxcuis = resolve_drug_to_ingredients(drug_name)
    rows = []
    if not ingredients:
        return [{'drug': drug_name, 'resolution_status': status, 'source_rxcuis': ';'.join(source_rxcuis), 'ingredient_rxcui': None, 'ingredient_name': None, 'ingredient_tty': None, 'ATC_code': None, 'ATC2': None, 'ATC3': None, 'ATC4': None}]
    for ingredient in ingredients:
        ingredient_rxcui = ingredient['ingredient_rxcui']
        atc_codes = direct_atc_codes_by_rxcui(ingredient_rxcui)
        if not atc_codes:
            rows.append({'drug': drug_name, 'resolution_status': 'ingredient_no_atc', 'source_rxcuis': ';'.join(source_rxcuis), 'ingredient_rxcui': ingredient_rxcui, 'ingredient_name': ingredient['ingredient_name'], 'ingredient_tty': ingredient['ingredient_tty'], 'ATC_code': None, 'ATC2': None, 'ATC3': None, 'ATC4': None})
            continue
        for code in atc_codes:
            rows.append({'drug': drug_name, 'resolution_status': 'mapped', 'source_rxcuis': ';'.join(source_rxcuis), 'ingredient_rxcui': ingredient_rxcui, 'ingredient_name': ingredient['ingredient_name'], 'ingredient_tty': ingredient['ingredient_tty'], 'ATC_code': code, 'ATC2': code[:3] if len(code) >= 3 else None, 'ATC3': code[:4] if len(code) >= 4 else None, 'ATC4': code[:5] if len(code) >= 5 else None})
    return rows


### 3.3 Recovery utilities

These helpers support PIN/MIN-to-IN recovery and the later NDC fallback without changing the primary text-mapping rule.


In [ ]:
# Recovery utilities for PIN/MIN concepts and NDC-derived RxNorm products.
def clean_rxcui(value):
    """
    Normalize RxCUI values read from CSV.
    Handles values such as 203191.0 caused by nullable columns.
    """
    if pd.isna(value):
        return None
    value = str(value).strip()
    if value.endswith('.0'):
        value = value[:-2]
    return value

def get_related_in_concepts(rxcui):
    """
    Retrieve RxNorm Ingredient (IN) concepts related to
    a PIN or MIN concept.
    """
    rxcui = clean_rxcui(rxcui)
    payload = get_json(f'{RXNAV_BASE}/rxcui/{rxcui}/related.json', params={'tty': 'IN'})
    groups = payload.get('relatedGroup', {}).get('conceptGroup', []) or []
    rows = []
    for group in groups:
        if group.get('tty') != 'IN':
            continue
        concepts = group.get('conceptProperties', []) or []
        for concept in concepts:
            rows.append({'parent_in_rxcui': clean_rxcui(concept.get('rxcui')), 'parent_in_name': concept.get('name')})
    if rows:
        rows = pd.DataFrame(rows).drop_duplicates().to_dict('records')
    return rows

def get_in_concepts_for_any_rxcui(rxcui):
    """
    Return RxNorm Ingredient (IN) concepts related to
    any active RxNorm concept/product.

    Examples:
        SCD/SBD/product -> IN
        PIN -> IN
        MIN -> constituent INs
        IN -> itself
    """
    rxcui = clean_rxcui(rxcui)
    if not rxcui:
        return []
    payload = get_json(f'{RXNAV_BASE}/rxcui/{rxcui}/related.json', params={'tty': 'IN'})
    groups = payload.get('relatedGroup', {}).get('conceptGroup', []) or []
    rows = []
    for group in groups:
        if group.get('tty') != 'IN':
            continue
        concepts = group.get('conceptProperties', []) or []
        for concept in concepts:
            ing_rxcui = clean_rxcui(concept.get('rxcui'))
            ing_name = concept.get('name')
            if ing_rxcui:
                rows.append({'rxcui': ing_rxcui, 'name': ing_name})
    unique = {}
    for row in rows:
        unique[row['rxcui']] = row
    return list(unique.values())

def get_ndc_status(ndc):
    """
    Query RxNorm getNDCStatus.

    history=0:
        return all available history records

    altpkg=1:
        if the exact package NDC is unknown, RxNorm may
        return an alternate packaging of the same product.
    """
    ndc = str(ndc).strip()
    payload = get_json(f'{RXNAV_BASE}/ndcstatus.json', params={'ndc': ndc, 'history': 0, 'altpkg': 1})
    return payload.get('ndcStatus', {}) or {}


## 4. Full text-based medication mapping

Each final-cohort medication name is mapped once and persisted to a checkpoint. The cell supports resumption and retries only entries recorded as API failures. This keeps transient service failures separate from genuine unmapped medication names.


In [ ]:
FULL_ATC_LONG_CSV = os.path.join(REDUCTION_DIR, 'drug_direct_atc_mapping_long.csv')
all_drugs = drug_prevalence['drug'].astype(str).tolist()
if os.path.exists(FULL_ATC_LONG_CSV):
    mapping_long = pd.read_csv(FULL_ATC_LONG_CSV)
    already_done = set(mapping_long['drug'].dropna().astype(str).unique())
    print('Existing mapping cache found:', len(already_done), 'drugs already processed')
else:
    mapping_long = pd.DataFrame()
    already_done = set()
pending_rows = []
for number, drug in enumerate(all_drugs, start=1):
    if drug in already_done:
        continue
    try:
        drug_rows = map_drug_direct_atc(drug)
        pending_rows.extend(drug_rows)
    except Exception as e:
        pending_rows.append({'drug': drug, 'resolution_status': 'api_error', 'source_rxcuis': None, 'ingredient_rxcui': None, 'ingredient_name': None, 'ingredient_tty': None, 'ATC_code': None, 'ATC2': None, 'ATC3': None, 'ATC4': None, 'error': repr(e)})
    time.sleep(0.1)
    processed_since_save = len({row['drug'] for row in pending_rows})
    if processed_since_save >= 100:
        new_df = pd.DataFrame(pending_rows)
        if len(mapping_long):
            mapping_long = pd.concat([mapping_long, new_df], ignore_index=True)
        else:
            mapping_long = new_df
        mapping_long.to_csv(FULL_ATC_LONG_CSV, index=False)
        already_done.update(new_df['drug'].dropna().astype(str).unique())
        pending_rows = []
        print(f'Processed {len(already_done)} of {len(all_drugs)} medication names')
if pending_rows:
    new_df = pd.DataFrame(pending_rows)
    if len(mapping_long):
        mapping_long = pd.concat([mapping_long, new_df], ignore_index=True)
    else:
        mapping_long = new_df
mapping_long.to_csv(FULL_ATC_LONG_CSV, index=False)
print()
print('Full mapping completed.')
print('Original drug names processed:', mapping_long['drug'].nunique())
print('Saved:', FULL_ATC_LONG_CSV)

mapping_long = pd.read_csv(FULL_ATC_LONG_CSV)
api_error_names = sorted(mapping_long.loc[mapping_long['resolution_status'].eq('api_error'), 'drug'].dropna().astype(str).unique())
print('API-error medication names to retry:', len(api_error_names))
mapping_clean = mapping_long.loc[~mapping_long['drug'].isin(api_error_names)].copy()
retry_rows = []
for i, drug in enumerate(api_error_names, start=1):
    try:
        drug_rows = map_drug_direct_atc(drug)
        retry_rows.extend(drug_rows)
    except Exception as e:
        retry_rows.append({'drug': drug, 'resolution_status': 'api_error', 'source_rxcuis': None, 'ingredient_rxcui': None, 'ingredient_name': None, 'ingredient_tty': None, 'ATC_code': None, 'ATC2': None, 'ATC3': None, 'ATC4': None, 'error': repr(e)})
    time.sleep(0.35)
    if i % 25 == 0:
        print(f'Retried {i} of {len(api_error_names)}')
retry_df = pd.DataFrame(retry_rows)
mapping_long = pd.concat([mapping_clean, retry_df], ignore_index=True)
mapping_long.to_csv(FULL_ATC_LONG_CSV, index=False)
remaining_api_errors = mapping_long.loc[mapping_long['resolution_status'].eq('api_error'), 'drug'].dropna().astype(str).nunique()
print()
print('======================================')
print('RETRY COMPLETE')
print('======================================')
print('Original API-error medication names:', len(api_error_names))
print('Remaining API-error drug names:', remaining_api_errors)
print('Saved mapping checkpoint:', FULL_ATC_LONG_CSV)


## 5. PIN/MIN recovery and text-mapping coverage

Some medication names resolve to precise or multi-ingredient RxNorm concepts that do not carry a direct ATC property. These concepts are resolved to parent/constituent IN concepts, after which direct ATC properties are queried. The recovered rows are appended to the mapping while the original rows are retained as provenance.


In [ ]:
def build_resolution_audit(mapping, prevalence):
    summary = mapping.groupby("drug").agg(
        n_ingredients=("ingredient_rxcui", lambda x: x.dropna().nunique()),
        n_ATC2=("ATC2", lambda x: x.dropna().nunique()),
        n_ATC3=("ATC3", lambda x: x.dropna().nunique()),
        statuses=("resolution_status", lambda x: "; ".join(sorted(set(x.dropna().astype(str))))),
        ingredient_names=("ingredient_name", lambda x: "; ".join(sorted(set(x.dropna().astype(str))))),
    ).reset_index()
    audit = prevalence.merge(summary, on="drug", how="left")
    audit[["n_ingredients", "n_ATC2", "n_ATC3"]] = audit[["n_ingredients", "n_ATC2", "n_ATC3"]].fillna(0)
    return audit

pre_recovery_audit = build_resolution_audit(mapping_long, drug_prevalence)
print("Medication names resolved to an ingredient before PIN/MIN recovery:", int((pre_recovery_audit['n_ingredients'] > 0).sum()))
print("Medication names with ATC3 before PIN/MIN recovery:", int((pre_recovery_audit['n_ATC3'] > 0).sum()))


In [ ]:
PIN_MIN_RECOVERY_CSV = os.path.join(REDUCTION_DIR, 'pin_min_to_in_atc_recovery.csv')
COMBINED_RECOVERED_CSV = os.path.join(REDUCTION_DIR, 'drug_direct_atc_mapping_long_recovered.csv')
resolved_no_atc_names = set(pre_recovery_audit.loc[(pre_recovery_audit['n_ingredients'] > 0) & (pre_recovery_audit['n_ATC3'] == 0), 'drug'].dropna().astype(str))
recovery_source = mapping_long.loc[mapping_long['drug'].isin(resolved_no_atc_names) & mapping_long['ingredient_tty'].isin(['PIN', 'MIN']), ['drug', 'ingredient_rxcui', 'ingredient_name', 'ingredient_tty']].drop_duplicates().copy()
recovery_source['ingredient_rxcui'] = recovery_source['ingredient_rxcui'].apply(clean_rxcui)
print('======================================')
print('PIN / MIN RECOVERY SOURCE')
print('======================================')
print('Drug names eligible:', recovery_source['drug'].nunique())
print('Unique PIN/MIN source concepts:', recovery_source['ingredient_rxcui'].nunique())
print()
display(recovery_source.head(20))
in_cache = {}
atc_cache = {}
recovery_rows = []
for i, row in enumerate(recovery_source.itertuples(index=False), start=1):
    drug = row.drug
    source_rxcui = clean_rxcui(row.ingredient_rxcui)
    source_name = row.ingredient_name
    source_tty = row.ingredient_tty
    try:
        if source_rxcui not in in_cache:
            in_cache[source_rxcui] = get_related_in_concepts(source_rxcui)
            time.sleep(0.2)
        related_ins = in_cache[source_rxcui]
    except Exception as e:
        recovery_rows.append({'drug': drug, 'source_rxcui': source_rxcui, 'source_name': source_name, 'source_tty': source_tty, 'ingredient_rxcui': None, 'ingredient_name': None, 'ingredient_tty': 'IN', 'ATC_code': None, 'ATC2': None, 'ATC3': None, 'ATC4': None, 'resolution_status': 'pin_min_in_lookup_error', 'error': repr(e)})
        continue
    if not related_ins:
        recovery_rows.append({'drug': drug, 'source_rxcui': source_rxcui, 'source_name': source_name, 'source_tty': source_tty, 'ingredient_rxcui': None, 'ingredient_name': None, 'ingredient_tty': 'IN', 'ATC_code': None, 'ATC2': None, 'ATC3': None, 'ATC4': None, 'resolution_status': 'pin_min_no_related_in'})
        continue
    for ing in related_ins:
        in_rxcui = clean_rxcui(ing['parent_in_rxcui'])
        in_name = ing['parent_in_name']
        try:
            if in_rxcui not in atc_cache:
                atc_cache[in_rxcui] = direct_atc_codes_by_rxcui(in_rxcui)
                time.sleep(0.2)
            codes = atc_cache[in_rxcui]
        except Exception as e:
            recovery_rows.append({'drug': drug, 'source_rxcui': source_rxcui, 'source_name': source_name, 'source_tty': source_tty, 'ingredient_rxcui': in_rxcui, 'ingredient_name': in_name, 'ingredient_tty': 'IN', 'ATC_code': None, 'ATC2': None, 'ATC3': None, 'ATC4': None, 'resolution_status': 'pin_min_in_atc_error', 'error': repr(e)})
            continue
        if not codes:
            recovery_rows.append({'drug': drug, 'source_rxcui': source_rxcui, 'source_name': source_name, 'source_tty': source_tty, 'ingredient_rxcui': in_rxcui, 'ingredient_name': in_name, 'ingredient_tty': 'IN', 'ATC_code': None, 'ATC2': None, 'ATC3': None, 'ATC4': None, 'resolution_status': 'pin_min_in_no_atc'})
            continue
        for code in codes:
            recovery_rows.append({'drug': drug, 'source_rxcui': source_rxcui, 'source_name': source_name, 'source_tty': source_tty, 'ingredient_rxcui': in_rxcui, 'ingredient_name': in_name, 'ingredient_tty': 'IN', 'ATC_code': code, 'ATC2': code[:3] if len(code) >= 3 else None, 'ATC3': code[:4] if len(code) >= 4 else None, 'ATC4': code[:5] if len(code) >= 5 else None, 'resolution_status': 'recovered_via_IN'})
    if i % 25 == 0:
        print(f'Processed {i} of {len(recovery_source)} PIN/MIN source rows')
pin_min_recovery = pd.DataFrame(recovery_rows).drop_duplicates()
pin_min_recovery.to_csv(PIN_MIN_RECOVERY_CSV, index=False)
print()
print('======================================')
print('PIN / MIN RECOVERY COMPLETE')
print('======================================')
print('Recovery rows:', len(pin_min_recovery))
print('Drug names attempted:', pin_min_recovery['drug'].nunique())
print('Drug names recovered with ATC3:', pin_min_recovery.loc[pin_min_recovery['ATC3'].notna(), 'drug'].nunique())
print('Distinct recovered ATC3 categories:', pin_min_recovery['ATC3'].dropna().nunique())
print('Saved recovery:', PIN_MIN_RECOVERY_CSV)
COMBINED_RECOVERED_CSV = os.path.join(REDUCTION_DIR, 'drug_direct_atc_mapping_long_recovered.csv')
recovery_for_merge = pin_min_recovery.copy()
for col in mapping_long.columns:
    if col not in recovery_for_merge.columns:
        recovery_for_merge[col] = None
recovery_for_merge = recovery_for_merge[mapping_long.columns].copy()
mapping_long_recovered = pd.concat([mapping_long, recovery_for_merge], ignore_index=True).drop_duplicates()
mapping_long_recovered.to_csv(COMBINED_RECOVERED_CSV, index=False)
mapping_long = mapping_long_recovered.copy()
print('======================================')
print('COMBINED MAPPING')
print('======================================')
print('Original medication names:', mapping_long['drug'].nunique())
print('Rows in combined mapping:', len(mapping_long))
print('Drug names with >=1 ATC3:', mapping_long.loc[mapping_long['ATC3'].notna(), 'drug'].nunique())
print('Distinct ATC3 categories:', mapping_long['ATC3'].dropna().nunique())
print()
print('Saved combined mapping:', COMBINED_RECOVERED_CSV)


In [ ]:
resolution_audit = build_resolution_audit(mapping_long, drug_prevalence)
has_ingredient = resolution_audit["n_ingredients"] > 0
has_atc3 = resolution_audit["n_ATC3"] > 0
total_exposures = resolution_audit["stay_count"].sum()
ingredient_exposures = resolution_audit.loc[has_ingredient, "stay_count"].sum()
atc3_exposures = resolution_audit.loc[has_atc3, "stay_count"].sum()
resolved_no_atc = has_ingredient & ~has_atc3
no_ingredient = ~has_ingredient

print("Drug names resolved to >=1 ingredient:", int(has_ingredient.sum()))
print(f"Ingredient exposure-weighted coverage: {ingredient_exposures / total_exposures:.2%}")
print("Drug names with ATC3:", int(has_atc3.sum()))
print(f"ATC3 exposure-weighted coverage: {atc3_exposures / total_exposures:.2%}")
print("Resolved ingredient but no ATC3:", int(resolved_no_atc.sum()))
print(f"Exposure share, resolved/no ATC3: {resolution_audit.loc[resolved_no_atc, 'stay_count'].sum() / total_exposures:.2%}")
print("No ingredient resolution:", int(no_ingredient.sum()))
print(f"Exposure share, no ingredient: {resolution_audit.loc[no_ingredient, 'stay_count'].sum() / total_exposures:.2%}")


## 6. NDC fallback for unresolved medication names

NDC (National Drug Code) is used only for medication names with no ingredient resolution after text-based mapping. Because the processed pharmacy checkpoint does not contain NDC or `hadm_id`, candidate NDCs are reconstructed from raw `prescriptions.csv.gz` by joining prescriptions to ICU stays through `subject_id` and `hadm_id`, then matching the resulting stay-drug pairs to the exact baseline cohort. This is an admission-to-ICU reconstruction and should not be interpreted as exact event-level linkage.


In [ ]:
mimic_candidates = [os.path.join(PROJECT_DIR, 'mimic-iv-3.1'), os.path.join(PROJECT_DIR, 'mimic-iv-3.1', 'mimic-iv-3.1')]
MIMIC_DIR = None
for candidate in mimic_candidates:
    prescriptions_test = os.path.join(candidate, 'hosp', 'prescriptions.csv.gz')
    icustays_test = os.path.join(candidate, 'icu', 'icustays.csv.gz')
    if os.path.exists(prescriptions_test) and os.path.exists(icustays_test):
        MIMIC_DIR = candidate
        break
if MIMIC_DIR is None:
    raise FileNotFoundError('Could not locate the raw MIMIC-IV prescriptions.csv.gz and icustays.csv.gz files.')
PRESCRIPTIONS_PATH = os.path.join(MIMIC_DIR, 'hosp', 'prescriptions.csv.gz')
ICUSTAYS_PATH = os.path.join(MIMIC_DIR, 'icu', 'icustays.csv.gz')
print('MIMIC directory:', MIMIC_DIR)
print('Prescriptions exists:', os.path.exists(PRESCRIPTIONS_PATH))
print('ICU stays exists:', os.path.exists(ICUSTAYS_PATH))
unresolved_drugs_df = resolution_audit.loc[resolution_audit['n_ingredients'] == 0, ['drug']].dropna().drop_duplicates().copy()
print()
print('Unresolved medication names:', len(unresolved_drugs_df))
pharm_path = PROCESSED_PHARM.replace('\\', '/')
prescriptions_path = PRESCRIPTIONS_PATH.replace('\\', '/')
icustays_path = ICUSTAYS_PATH.replace('\\', '/')
con = duckdb.connect()
con.execute("PRAGMA memory_limit='12GB'")
con.register('unresolved_drugs_df', unresolved_drugs_df)
if os.path.exists(NOIMP_PARQUET):
    cohort_path = NOIMP_PARQUET.replace('\\', '/')
    con.execute(f"\n        CREATE OR REPLACE TEMP VIEW final_cohort AS\n\n        SELECT DISTINCT\n            CAST(subject_id AS BIGINT) AS subject_id,\n            CAST(stay_id AS BIGINT) AS stay_id\n\n        FROM read_parquet(\n            '{cohort_path}'\n        )\n        ")
else:
    cohort_path = NOIMP_CSV.replace('\\', '/')
    con.execute(f"\n        CREATE OR REPLACE TEMP VIEW final_cohort AS\n\n        SELECT DISTINCT\n            TRY_CAST(subject_id AS BIGINT) AS subject_id,\n            TRY_CAST(stay_id AS BIGINT) AS stay_id\n\n        FROM read_csv_auto(\n            '{cohort_path}',\n            header=true\n        )\n        ")
con.execute(f"\n    CREATE OR REPLACE TEMP TABLE\n    baseline_unresolved_presence AS\n\n    SELECT DISTINCT\n\n        CAST(p.subject_id AS BIGINT)\n            AS subject_id,\n\n        CAST(p.stay_id AS BIGINT)\n            AS stay_id,\n\n        TRIM(\n            CAST(p.drug AS VARCHAR)\n        ) AS drug\n\n    FROM read_parquet(\n        '{pharm_path}'\n    ) AS p\n\n    INNER JOIN final_cohort AS c\n\n        ON\n        CAST(p.subject_id AS BIGINT)\n            = c.subject_id\n\n        AND\n\n        CAST(p.stay_id AS BIGINT)\n            = c.stay_id\n\n    INNER JOIN unresolved_drugs_df AS u\n\n        ON TRIM(\n            CAST(p.drug AS VARCHAR)\n        ) = u.drug\n\n    WHERE\n        p.drug IS NOT NULL\n    ")
baseline_unresolved_count = con.execute('\n        SELECT COUNT(*)\n        FROM baseline_unresolved_presence\n        ').fetchone()[0]
print()
print('Unresolved stay-drug presences in exact baseline:', baseline_unresolved_count)
con.execute(f"\n    CREATE OR REPLACE TEMP TABLE\n    raw_prescription_ndc AS\n\n    SELECT DISTINCT\n\n        TRY_CAST(\n            p.subject_id AS BIGINT\n        ) AS subject_id,\n\n        TRY_CAST(\n            i.stay_id AS BIGINT\n        ) AS stay_id,\n\n        TRIM(\n            CAST(p.drug AS VARCHAR)\n        ) AS drug,\n\n        TRIM(\n            CAST(p.ndc AS VARCHAR)\n        ) AS ndc_raw,\n\n        regexp_replace(\n            TRIM(\n                CAST(p.ndc AS VARCHAR)\n            ),\n            '[^0-9]',\n            '',\n            'g'\n        ) AS ndc_digits\n\n    FROM read_csv_auto(\n        '{prescriptions_path}',\n        header=true,\n        all_varchar=true,\n        union_by_name=true\n    ) AS p\n\n    INNER JOIN read_csv_auto(\n        '{icustays_path}',\n        header=true,\n        all_varchar=true,\n        union_by_name=true\n    ) AS i\n\n        ON\n        TRY_CAST(\n            p.subject_id AS BIGINT\n        )\n        =\n        TRY_CAST(\n            i.subject_id AS BIGINT\n        )\n\n        AND\n\n        TRY_CAST(\n            p.hadm_id AS BIGINT\n        )\n        =\n        TRY_CAST(\n            i.hadm_id AS BIGINT\n        )\n\n    INNER JOIN final_cohort AS c\n\n        ON\n        TRY_CAST(\n            i.subject_id AS BIGINT\n        ) = c.subject_id\n\n        AND\n\n        TRY_CAST(\n            i.stay_id AS BIGINT\n        ) = c.stay_id\n\n    INNER JOIN unresolved_drugs_df AS u\n\n        ON TRIM(\n            CAST(p.drug AS VARCHAR)\n        ) = u.drug\n\n    WHERE\n        p.drug IS NOT NULL\n    ")
ndc_stay_drug = con.execute("\n        SELECT\n\n            b.subject_id,\n            b.stay_id,\n            b.drug,\n\n            MAX(\n                CASE\n\n                    WHEN\n                        r.ndc_digits IS NOT NULL\n\n                        AND\n                        r.ndc_digits <> ''\n\n                        AND\n                        regexp_replace(\n                            r.ndc_digits,\n                            '0',\n                            '',\n                            'g'\n                        ) <> ''\n\n                    THEN 1\n\n                    ELSE 0\n\n                END\n            ) AS has_candidate_ndc,\n\n            COUNT(\n                DISTINCT\n                CASE\n\n                    WHEN\n                        r.ndc_digits IS NOT NULL\n\n                        AND\n                        r.ndc_digits <> ''\n\n                        AND\n                        regexp_replace(\n                            r.ndc_digits,\n                            '0',\n                            '',\n                            'g'\n                        ) <> ''\n\n                    THEN r.ndc_digits\n\n                    ELSE NULL\n\n                END\n            ) AS n_candidate_ndcs\n\n        FROM baseline_unresolved_presence AS b\n\n        LEFT JOIN raw_prescription_ndc AS r\n\n            ON\n            b.subject_id = r.subject_id\n\n            AND\n            b.stay_id = r.stay_id\n\n            AND\n            b.drug = r.drug\n\n        GROUP BY\n\n            b.subject_id,\n            b.stay_id,\n            b.drug\n        ").fetchdf()
ndc_drug_summary = ndc_stay_drug.groupby('drug', as_index=False).agg(baseline_stays=('stay_id', 'size'), stays_with_candidate_ndc=('has_candidate_ndc', 'sum'), max_ndcs_per_stay_drug=('n_candidate_ndcs', 'max'))
ndc_drug_summary['ndc_stay_coverage'] = ndc_drug_summary['stays_with_candidate_ndc'] / ndc_drug_summary['baseline_stays']
total_unresolved_presence = len(ndc_stay_drug)
presence_with_ndc = ndc_stay_drug['has_candidate_ndc'].sum()
drug_names_with_ndc = (ndc_drug_summary['stays_with_candidate_ndc'] > 0).sum()
print()
print('======================================')
print('NDC AVAILABILITY — UNRESOLVED DRUGS')
print('======================================')
print('Unresolved medication names:', len(unresolved_drugs_df))
print('Names with >=1 candidate NDC:', drug_names_with_ndc)
print('Drug-name NDC availability:', f'{drug_names_with_ndc / len(unresolved_drugs_df):.2%}')
print()
print('Exact unresolved stay-drug presences:', total_unresolved_presence)
print('Stay-drug presences with candidate NDC:', int(presence_with_ndc))
print('Stay-drug NDC availability:', f'{presence_with_ndc / total_unresolved_presence:.2%}')
ndc_top = resolution_audit.loc[resolution_audit['n_ingredients'] == 0, ['drug', 'stay_count', 'prevalence']].merge(ndc_drug_summary, on='drug', how='left').sort_values('stay_count', ascending=False)
print()
print('Most prevalent unresolved drugs and NDC availability:')
display(ndc_top.head(20))
ndc_length_audit = con.execute("\n        SELECT\n\n            LENGTH(ndc_digits)\n                AS digit_length,\n\n            COUNT(DISTINCT ndc_digits)\n                AS unique_ndcs,\n\n            COUNT(*)\n                AS prescription_rows\n\n        FROM raw_prescription_ndc\n\n        WHERE\n            ndc_digits IS NOT NULL\n\n            AND\n            ndc_digits <> ''\n\n            AND\n            regexp_replace(\n                ndc_digits,\n                '0',\n                '',\n                'g'\n            ) <> ''\n\n        GROUP BY\n            LENGTH(ndc_digits)\n\n        ORDER BY\n            digit_length\n        ").fetchdf()
print()
print('Candidate NDC digit-length distribution:')
display(ndc_length_audit)
NDC_AVAILABILITY_CSV = os.path.join(REDUCTION_DIR, 'unresolved_drug_ndc_availability.csv')
ndc_top.to_csv(NDC_AVAILABILITY_CSV, index=False)
print()
print('Saved NDC availability audit:', NDC_AVAILABILITY_CSV)
con.close()


### 6.1 NDC-to-RxNorm status and ATC recovery

Candidate NDCs are checked against RxNorm. Only records with an active RxNorm concept are used for the primary recovery step. Product concepts are resolved to ingredient concepts and then to direct ATC properties.


In [ ]:
FULL_NDC_LIST_CSV = os.path.join(REDUCTION_DIR, 'unresolved_candidate_ndcs_full.csv')
FULL_NDC_STATUS_CSV = os.path.join(REDUCTION_DIR, 'ndc_rxnorm_status_full.csv')
unresolved_drugs_df = resolution_audit.loc[resolution_audit['n_ingredients'] == 0, ['drug']].dropna().drop_duplicates().copy()
print('Unresolved medication names:', len(unresolved_drugs_df))
pharm_path = PROCESSED_PHARM.replace('\\', '/')
prescriptions_path = PRESCRIPTIONS_PATH.replace('\\', '/')
icustays_path = ICUSTAYS_PATH.replace('\\', '/')
con = duckdb.connect()
con.execute("PRAGMA memory_limit='12GB'")
con.register('unresolved_drugs_df', unresolved_drugs_df)
if os.path.exists(NOIMP_PARQUET):
    cohort_path = NOIMP_PARQUET.replace('\\', '/')
    con.execute(f"\n        CREATE OR REPLACE TEMP VIEW final_cohort AS\n\n        SELECT DISTINCT\n            CAST(subject_id AS BIGINT) AS subject_id,\n            CAST(stay_id AS BIGINT) AS stay_id\n\n        FROM read_parquet(\n            '{cohort_path}'\n        )\n        ")
else:
    cohort_path = NOIMP_CSV.replace('\\', '/')
    con.execute(f"\n        CREATE OR REPLACE TEMP VIEW final_cohort AS\n\n        SELECT DISTINCT\n            TRY_CAST(subject_id AS BIGINT) AS subject_id,\n            TRY_CAST(stay_id AS BIGINT) AS stay_id\n\n        FROM read_csv_auto(\n            '{cohort_path}',\n            header=true\n        )\n        ")
con.execute(f"\n    CREATE OR REPLACE TEMP TABLE\n    baseline_unresolved_presence AS\n\n    SELECT DISTINCT\n\n        CAST(p.subject_id AS BIGINT)\n            AS subject_id,\n\n        CAST(p.stay_id AS BIGINT)\n            AS stay_id,\n\n        TRIM(\n            CAST(p.drug AS VARCHAR)\n        ) AS drug\n\n    FROM read_parquet(\n        '{pharm_path}'\n    ) AS p\n\n    INNER JOIN final_cohort AS c\n\n        ON CAST(p.subject_id AS BIGINT)\n            = c.subject_id\n\n        AND CAST(p.stay_id AS BIGINT)\n            = c.stay_id\n\n    INNER JOIN unresolved_drugs_df AS u\n\n        ON TRIM(\n            CAST(p.drug AS VARCHAR)\n        ) = u.drug\n    ")
full_drug_ndcs = con.execute(f"\n    WITH raw_ndcs AS (\n\n        SELECT\n\n            TRY_CAST(\n                p.subject_id AS BIGINT\n            ) AS subject_id,\n\n            TRY_CAST(\n                i.stay_id AS BIGINT\n            ) AS stay_id,\n\n            TRIM(\n                CAST(p.drug AS VARCHAR)\n            ) AS drug,\n\n            regexp_replace(\n                TRIM(\n                    CAST(p.ndc AS VARCHAR)\n                ),\n                '[^0-9]',\n                '',\n                'g'\n            ) AS ndc\n\n        FROM read_csv_auto(\n            '{prescriptions_path}',\n            header=true,\n            all_varchar=true,\n            union_by_name=true\n        ) AS p\n\n        INNER JOIN read_csv_auto(\n            '{icustays_path}',\n            header=true,\n            all_varchar=true,\n            union_by_name=true\n        ) AS i\n\n            ON TRY_CAST(\n                p.subject_id AS BIGINT\n            ) = TRY_CAST(\n                i.subject_id AS BIGINT\n            )\n\n            AND TRY_CAST(\n                p.hadm_id AS BIGINT\n            ) = TRY_CAST(\n                i.hadm_id AS BIGINT\n            )\n    )\n\n    SELECT\n\n        r.drug,\n        r.ndc,\n        COUNT(*) AS n_rows\n\n    FROM raw_ndcs AS r\n\n    INNER JOIN baseline_unresolved_presence AS b\n\n        ON r.subject_id = b.subject_id\n\n        AND r.stay_id = b.stay_id\n\n        AND r.drug = b.drug\n\n    WHERE\n        r.ndc IS NOT NULL\n\n        AND r.ndc <> ''\n\n        AND LENGTH(r.ndc) = 11\n\n        AND regexp_replace(\n            r.ndc,\n            '0',\n            '',\n            'g'\n        ) <> ''\n\n    GROUP BY\n        r.drug,\n        r.ndc\n\n    ORDER BY\n        r.drug,\n        n_rows DESC\n    ").fetchdf()
con.close()
full_drug_ndcs['ndc'] = full_drug_ndcs['ndc'].astype(str).str.zfill(11)
full_drug_ndcs.to_csv(FULL_NDC_LIST_CSV, index=False)
unique_ndcs = sorted(full_drug_ndcs['ndc'].dropna().unique())
print()
print('======================================')
print('FULL NDC CANDIDATE SET')
print('======================================')
print('Drug names represented:', full_drug_ndcs['drug'].nunique())
print('Unique candidate NDCs:', len(unique_ndcs))
print('Saved candidate list:', FULL_NDC_LIST_CSV)
if os.path.exists(FULL_NDC_STATUS_CSV):
    previous = pd.read_csv(FULL_NDC_STATUS_CSV, dtype={'input_ndc': str, 'returned_ndc': str, 'main_rxcui': str})
    previous['input_ndc'] = previous['input_ndc'].astype(str).str.zfill(11)
    completed_ndcs = set(previous.loc[previous['ndc_status'].fillna('').ne('API_ERROR'), 'input_ndc'])
    status_rows = previous.loc[previous['input_ndc'].isin(completed_ndcs)].to_dict('records')
    print()
    print('Resuming checkpoint.')
    print('Already completed:', len(completed_ndcs))
else:
    completed_ndcs = set()
    status_rows = []
remaining_ndcs = [ndc for ndc in unique_ndcs if ndc not in completed_ndcs]
print('NDCs remaining:', len(remaining_ndcs))

def save_ndc_status_checkpoint(rows):
    checkpoint = pd.DataFrame(rows).drop_duplicates(subset=['input_ndc'], keep='last')
    checkpoint.to_csv(FULL_NDC_STATUS_CSV, index=False)
    return checkpoint
for i, ndc in enumerate(remaining_ndcs, start=1):
    try:
        status_data = get_ndc_status(ndc)
        ndc_status = status_data.get('status')
        returned_ndc = status_data.get('ndc11')
        alt_ndc = status_data.get('altNdc')
        main_rxcui = status_data.get('rxcui')
        concept_name = status_data.get('conceptName')
        concept_status = status_data.get('conceptStatus')
        history = status_data.get('ndcHistory', []) or []
        if isinstance(history, dict):
            history = [history]
        active_rxcuis = []
        original_rxcuis = []
        for h in history:
            active_rxcui = h.get('activeRxcui')
            original_rxcui = h.get('originalRxcui')
            if active_rxcui:
                active_rxcuis.append(str(active_rxcui))
            if original_rxcui:
                original_rxcuis.append(str(original_rxcui))
        active_rxcuis = list(dict.fromkeys(active_rxcuis))
        original_rxcuis = list(dict.fromkeys(original_rxcuis))
        status_rows.append({'input_ndc': ndc, 'returned_ndc': returned_ndc, 'ndc_status': ndc_status, 'alt_ndc': alt_ndc, 'main_rxcui': main_rxcui, 'concept_name': concept_name, 'concept_status': concept_status, 'active_rxcuis': ';'.join(active_rxcuis), 'original_rxcuis': ';'.join(original_rxcuis), 'n_history_records': len(history), 'error': None})
    except Exception as e:
        status_rows.append({'input_ndc': ndc, 'returned_ndc': None, 'ndc_status': 'API_ERROR', 'alt_ndc': None, 'main_rxcui': None, 'concept_name': None, 'concept_status': None, 'active_rxcuis': None, 'original_rxcuis': None, 'n_history_records': None, 'error': repr(e)})
    if i % 50 == 0:
        save_ndc_status_checkpoint(status_rows)
        print(f'Processed {i} of {len(remaining_ndcs)} remaining NDCs')
    time.sleep(0.25)
full_ndc_status = save_ndc_status_checkpoint(status_rows)
print()
print('======================================')
print('FULL NDC -> RXNORM STATUS COMPLETE')
print('======================================')
print('Unique candidate NDCs:', len(unique_ndcs))
print('NDC status rows:', len(full_ndc_status))
print()
print('NDC status distribution:')
display(full_ndc_status['ndc_status'].value_counts(dropna=False).rename('count').to_frame())
print()
print('Concept status distribution:')
display(full_ndc_status['concept_status'].value_counts(dropna=False).rename('count').to_frame())
usable_ndcs = full_ndc_status['main_rxcui'].notna() & full_ndc_status['concept_status'].eq('ACTIVE')
print()
print('NDCs with active RxNorm product:', int(usable_ndcs.sum()))
print('Active-product coverage of NDCs:', f'{usable_ndcs.mean():.2%}')
print()
print('Saved full NDC status mapping:', FULL_NDC_STATUS_CSV)


In [ ]:
FULL_NDC_ATC_CSV = os.path.join(REDUCTION_DIR, 'ndc_active_product_to_atc_full.csv')
if 'full_ndc_status' not in globals():
    full_ndc_status = pd.read_csv(FULL_NDC_STATUS_CSV, dtype={'input_ndc': str, 'main_rxcui': str})
full_ndc_status['input_ndc'] = full_ndc_status['input_ndc'].astype(str).str.zfill(11)
usable_status = full_ndc_status.loc[full_ndc_status['concept_status'].eq('ACTIVE') & full_ndc_status['main_rxcui'].notna()].copy()
usable_status['main_rxcui'] = usable_status['main_rxcui'].apply(clean_rxcui)
print('======================================')
print('FULL ACTIVE NDC RECOVERY SOURCE')
print('======================================')
print('Usable NDCs:', usable_status['input_ndc'].nunique())
print('Unique product RxCUIs:', usable_status['main_rxcui'].nunique())
ndc_drug_source = full_drug_ndcs[['drug', 'ndc']].drop_duplicates().merge(usable_status[['input_ndc', 'ndc_status', 'main_rxcui', 'concept_name', 'concept_status']], left_on='ndc', right_on='input_ndc', how='inner')
print('Drug names represented:', ndc_drug_source['drug'].nunique())
product_in_cache = {}
unique_products = ndc_drug_source['main_rxcui'].dropna().unique()
for i, product_rxcui in enumerate(unique_products, start=1):
    try:
        product_in_cache[product_rxcui] = get_in_concepts_for_any_rxcui(product_rxcui)
    except Exception as e:
        product_in_cache[product_rxcui] = {'__error__': repr(e)}
    if i % 50 == 0:
        print(f'Resolved IN concepts for {i} of {len(unique_products)} product RxCUIs')
    time.sleep(0.2)
ingredient_ids = set()
for result in product_in_cache.values():
    if isinstance(result, dict):
        continue
    for ing in result:
        rxcui = clean_rxcui(ing.get('rxcui'))
        if rxcui:
            ingredient_ids.add(rxcui)
print()
print('Unique IN RxCUIs resolved:', len(ingredient_ids))
ingredient_atc_cache = {}
for i, ing_rxcui in enumerate(sorted(ingredient_ids), start=1):
    try:
        ingredient_atc_cache[ing_rxcui] = direct_atc_codes_by_rxcui(ing_rxcui)
    except Exception as e:
        ingredient_atc_cache[ing_rxcui] = {'__error__': repr(e)}
    if i % 50 == 0:
        print(f'Resolved ATC for {i} of {len(ingredient_ids)} ingredient RxCUIs')
    time.sleep(0.2)
full_ndc_atc_rows = []
for _, row in ndc_drug_source.iterrows():
    drug = row['drug']
    ndc = row['input_ndc']
    product_rxcui = clean_rxcui(row['main_rxcui'])
    product_name = row['concept_name']
    ndc_status = row['ndc_status']
    in_result = product_in_cache.get(product_rxcui, [])
    if isinstance(in_result, dict):
        full_ndc_atc_rows.append({'drug': drug, 'ndc': ndc, 'ndc_status': ndc_status, 'product_rxcui': product_rxcui, 'product_name': product_name, 'ingredient_rxcui': None, 'ingredient_name': None, 'ATC_code': None, 'ATC2': None, 'ATC3': None, 'ATC4': None, 'status': 'ingredient_lookup_error'})
        continue
    if not in_result:
        full_ndc_atc_rows.append({'drug': drug, 'ndc': ndc, 'ndc_status': ndc_status, 'product_rxcui': product_rxcui, 'product_name': product_name, 'ingredient_rxcui': None, 'ingredient_name': None, 'ATC_code': None, 'ATC2': None, 'ATC3': None, 'ATC4': None, 'status': 'no_IN_resolution'})
        continue
    for ing in in_result:
        ingredient_rxcui = clean_rxcui(ing['rxcui'])
        ingredient_name = ing['name']
        atc_result = ingredient_atc_cache.get(ingredient_rxcui, [])
        if isinstance(atc_result, dict):
            full_ndc_atc_rows.append({'drug': drug, 'ndc': ndc, 'ndc_status': ndc_status, 'product_rxcui': product_rxcui, 'product_name': product_name, 'ingredient_rxcui': ingredient_rxcui, 'ingredient_name': ingredient_name, 'ATC_code': None, 'ATC2': None, 'ATC3': None, 'ATC4': None, 'status': 'ATC_lookup_error'})
            continue
        if not atc_result:
            full_ndc_atc_rows.append({'drug': drug, 'ndc': ndc, 'ndc_status': ndc_status, 'product_rxcui': product_rxcui, 'product_name': product_name, 'ingredient_rxcui': ingredient_rxcui, 'ingredient_name': ingredient_name, 'ATC_code': None, 'ATC2': None, 'ATC3': None, 'ATC4': None, 'status': 'IN_no_ATC'})
            continue
        for code in atc_result:
            full_ndc_atc_rows.append({'drug': drug, 'ndc': ndc, 'ndc_status': ndc_status, 'product_rxcui': product_rxcui, 'product_name': product_name, 'ingredient_rxcui': ingredient_rxcui, 'ingredient_name': ingredient_name, 'ATC_code': code, 'ATC2': code[:3] if len(code) >= 3 else None, 'ATC3': code[:4] if len(code) >= 4 else None, 'ATC4': code[:5] if len(code) >= 5 else None, 'status': 'recovered_via_NDC'})
full_ndc_atc = pd.DataFrame(full_ndc_atc_rows).drop_duplicates()
full_ndc_atc.to_csv(FULL_NDC_ATC_CSV, index=False)
print()
print('======================================')
print('FULL ACTIVE-NDC -> IN -> ATC COMPLETE')
print('======================================')
print('Drug names attempted:', full_ndc_atc['drug'].nunique())
print('Drug names with IN resolution:', full_ndc_atc.loc[full_ndc_atc['ingredient_rxcui'].notna(), 'drug'].nunique())
print('Drug names recovered with ATC3:', full_ndc_atc.loc[full_ndc_atc['ATC3'].notna(), 'drug'].nunique())
print('Distinct ATC3 categories:', full_ndc_atc['ATC3'].dropna().nunique())
print()
print('Status distribution:')
display(full_ndc_atc['status'].value_counts(dropna=False).rename('count').to_frame())
print()
print('Saved full NDC -> ATC mapping:', FULL_NDC_ATC_CSV)


In [ ]:
successful_ndc_atc3 = full_ndc_atc.loc[full_ndc_atc['ATC3'].notna(), ['ndc', 'ATC3']].drop_duplicates().copy()
successful_ndc_atc3['ndc'] = successful_ndc_atc3['ndc'].astype(str).str.zfill(11)
print('NDCs with >=1 ATC3:', successful_ndc_atc3['ndc'].nunique())
print('Distinct recovered ATC3 categories:', successful_ndc_atc3['ATC3'].nunique())
unresolved_drugs_df = resolution_audit.loc[resolution_audit['n_ingredients'] == 0, ['drug']].dropna().drop_duplicates().copy()
pharm_path = PROCESSED_PHARM.replace('\\', '/')
prescriptions_path = PRESCRIPTIONS_PATH.replace('\\', '/')
icustays_path = ICUSTAYS_PATH.replace('\\', '/')
con = duckdb.connect()
con.execute("PRAGMA memory_limit='12GB'")
con.register('unresolved_drugs_df', unresolved_drugs_df)
con.register('successful_ndc_atc3', successful_ndc_atc3)
if os.path.exists(NOIMP_PARQUET):
    cohort_path = NOIMP_PARQUET.replace('\\', '/')
    con.execute(f"\n        CREATE OR REPLACE TEMP VIEW final_cohort AS\n\n        SELECT DISTINCT\n            CAST(subject_id AS BIGINT) AS subject_id,\n            CAST(stay_id AS BIGINT) AS stay_id\n\n        FROM read_parquet(\n            '{cohort_path}'\n        )\n        ")
else:
    cohort_path = NOIMP_CSV.replace('\\', '/')
    con.execute(f"\n        CREATE OR REPLACE TEMP VIEW final_cohort AS\n\n        SELECT DISTINCT\n            TRY_CAST(subject_id AS BIGINT) AS subject_id,\n            TRY_CAST(stay_id AS BIGINT) AS stay_id\n\n        FROM read_csv_auto(\n            '{cohort_path}',\n            header=true\n        )\n        ")
con.execute(f"\n    CREATE OR REPLACE TEMP TABLE\n    unresolved_baseline AS\n\n    SELECT DISTINCT\n\n        CAST(p.subject_id AS BIGINT)\n            AS subject_id,\n\n        CAST(p.stay_id AS BIGINT)\n            AS stay_id,\n\n        TRIM(\n            CAST(p.drug AS VARCHAR)\n        ) AS drug\n\n    FROM read_parquet(\n        '{pharm_path}'\n    ) AS p\n\n    INNER JOIN final_cohort AS c\n\n        ON CAST(p.subject_id AS BIGINT)\n            = c.subject_id\n\n        AND CAST(p.stay_id AS BIGINT)\n            = c.stay_id\n\n    INNER JOIN unresolved_drugs_df AS u\n\n        ON TRIM(\n            CAST(p.drug AS VARCHAR)\n        ) = u.drug\n    ")
con.execute(f"\n    CREATE OR REPLACE TEMP TABLE\n    raw_prescription_ndcs AS\n\n    SELECT DISTINCT\n\n        TRY_CAST(\n            p.subject_id AS BIGINT\n        ) AS subject_id,\n\n        TRY_CAST(\n            i.stay_id AS BIGINT\n        ) AS stay_id,\n\n        TRIM(\n            CAST(p.drug AS VARCHAR)\n        ) AS drug,\n\n        regexp_replace(\n            TRIM(\n                CAST(p.ndc AS VARCHAR)\n            ),\n            '[^0-9]',\n            '',\n            'g'\n        ) AS ndc\n\n    FROM read_csv_auto(\n        '{prescriptions_path}',\n        header=true,\n        all_varchar=true,\n        union_by_name=true\n    ) AS p\n\n    INNER JOIN read_csv_auto(\n        '{icustays_path}',\n        header=true,\n        all_varchar=true,\n        union_by_name=true\n    ) AS i\n\n        ON TRY_CAST(\n            p.subject_id AS BIGINT\n        ) = TRY_CAST(\n            i.subject_id AS BIGINT\n        )\n\n        AND TRY_CAST(\n            p.hadm_id AS BIGINT\n        ) = TRY_CAST(\n            i.hadm_id AS BIGINT\n        )\n\n    WHERE\n        p.ndc IS NOT NULL\n    ")
recovered_presence = con.execute('\n    SELECT DISTINCT\n\n        b.subject_id,\n        b.stay_id,\n        b.drug\n\n    FROM unresolved_baseline AS b\n\n    INNER JOIN raw_prescription_ndcs AS r\n\n        ON b.subject_id = r.subject_id\n\n        AND b.stay_id = r.stay_id\n\n        AND b.drug = r.drug\n\n    INNER JOIN successful_ndc_atc3 AS m\n\n        ON r.ndc = m.ndc\n    ').fetchdf()
baseline_counts = con.execute('\n    SELECT\n\n        COUNT(*) AS n_unresolved_presences,\n\n        COUNT(\n            DISTINCT drug\n        ) AS n_unresolved_drug_names\n\n    FROM unresolved_baseline\n    ').fetchdf()
con.close()
n_unresolved_presences = int(baseline_counts.loc[0, 'n_unresolved_presences'])
n_recovered_presences = len(recovered_presence)
recovery_rate = n_recovered_presences / n_unresolved_presences
PRE_NDC_ATC3_COVERAGE = atc3_exposures / total_exposures
PRE_NDC_NO_INGREDIENT_SHARE = resolution_audit.loc[resolution_audit['n_ingredients'] == 0, 'stay_count'].sum() / total_exposures
added_overall_coverage = PRE_NDC_NO_INGREDIENT_SHARE * recovery_rate
new_overall_atc3_coverage = PRE_NDC_ATC3_COVERAGE + added_overall_coverage
print()
print('======================================')
print('EXPOSURE-WEIGHTED NDC RECOVERY')
print('======================================')
print('Original unresolved stay-drug presences:', n_unresolved_presences)
print('Recovered unresolved stay-drug presences:', n_recovered_presences)
print('Recovery within previously unresolved exposure:', f'{recovery_rate:.2%}')
print()
print('Added overall ATC3 coverage:', f'{added_overall_coverage:.2%}')
print('Previous overall ATC3 coverage:', f'{PRE_NDC_ATC3_COVERAGE:.2%}')
print('Estimated new overall ATC3 coverage:', f'{new_overall_atc3_coverage:.2%}')


## 7. Final stay-level ATC3 representation

Text-derived ATC3 assignments are primary. NDC-derived assignments are added only for text-unresolved medication names. The final long representation is deduplicated at `(subject_id, stay_id, ATC3)` and then pivoted into binary medication indicators.


In [ ]:
FINAL_ATC3_LONG_PARQUET = os.path.join(REDUCTION_DIR, 'final_medication_atc3_stay_long.parquet')
FINAL_ATC3_AUDIT_CSV = os.path.join(REDUCTION_DIR, 'final_medication_atc3_coverage_audit.csv')
text_atc3 = mapping_long.loc[mapping_long['ATC3'].notna(), ['drug', 'ATC3']].drop_duplicates().copy()
print('Drug names with text-derived ATC3:', text_atc3['drug'].nunique())
print('Text-derived ATC3 categories:', text_atc3['ATC3'].nunique())
successful_ndc_atc3 = full_ndc_atc.loc[full_ndc_atc['ATC3'].notna(), ['ndc', 'ATC3']].drop_duplicates().copy()
successful_ndc_atc3['ndc'] = successful_ndc_atc3['ndc'].astype(str).str.zfill(11)
print('NDCs with ATC3:', successful_ndc_atc3['ndc'].nunique())
print('NDC-derived ATC3 categories:', successful_ndc_atc3['ATC3'].nunique())
unresolved_drugs_df = resolution_audit.loc[resolution_audit['n_ingredients'] == 0, ['drug']].dropna().drop_duplicates().copy()
pharm_path = PROCESSED_PHARM.replace('\\', '/')
prescriptions_path = PRESCRIPTIONS_PATH.replace('\\', '/')
icustays_path = ICUSTAYS_PATH.replace('\\', '/')
output_long_path = FINAL_ATC3_LONG_PARQUET.replace('\\', '/')
con = duckdb.connect()
con.execute("PRAGMA memory_limit='12GB'")
con.register('text_atc3', text_atc3)
con.register('successful_ndc_atc3', successful_ndc_atc3)
con.register('unresolved_drugs_df', unresolved_drugs_df)
if os.path.exists(NOIMP_PARQUET):
    cohort_path = NOIMP_PARQUET.replace('\\', '/')
    con.execute(f"\n        CREATE OR REPLACE TEMP VIEW final_cohort AS\n\n        SELECT DISTINCT\n\n            CAST(\n                subject_id AS BIGINT\n            ) AS subject_id,\n\n            CAST(\n                stay_id AS BIGINT\n            ) AS stay_id\n\n        FROM read_parquet(\n            '{cohort_path}'\n        )\n        ")
else:
    cohort_path = NOIMP_CSV.replace('\\', '/')
    con.execute(f"\n        CREATE OR REPLACE TEMP VIEW final_cohort AS\n\n        SELECT DISTINCT\n\n            TRY_CAST(\n                subject_id AS BIGINT\n            ) AS subject_id,\n\n            TRY_CAST(\n                stay_id AS BIGINT\n            ) AS stay_id\n\n        FROM read_csv_auto(\n            '{cohort_path}',\n            header=true\n        )\n        ")
con.execute(f"\n    CREATE OR REPLACE TEMP TABLE\n    medication_baseline AS\n\n    SELECT DISTINCT\n\n        CAST(\n            p.subject_id AS BIGINT\n        ) AS subject_id,\n\n        CAST(\n            p.stay_id AS BIGINT\n        ) AS stay_id,\n\n        TRIM(\n            CAST(\n                p.drug AS VARCHAR\n            )\n        ) AS drug\n\n    FROM read_parquet(\n        '{pharm_path}'\n    ) AS p\n\n    INNER JOIN final_cohort AS c\n\n        ON CAST(\n            p.subject_id AS BIGINT\n        ) = c.subject_id\n\n        AND CAST(\n            p.stay_id AS BIGINT\n        ) = c.stay_id\n    ")
con.execute("\n    CREATE OR REPLACE TEMP TABLE\n    text_stay_atc3 AS\n\n    SELECT DISTINCT\n\n        b.subject_id,\n        b.stay_id,\n        b.drug,\n        m.ATC3,\n        'text_rxnorm' AS mapping_source\n\n    FROM medication_baseline AS b\n\n    INNER JOIN text_atc3 AS m\n\n        ON b.drug = m.drug\n    ")
con.execute(f"\n    CREATE OR REPLACE TEMP TABLE\n    raw_stay_ndcs AS\n\n    SELECT DISTINCT\n\n        TRY_CAST(\n            p.subject_id AS BIGINT\n        ) AS subject_id,\n\n        TRY_CAST(\n            i.stay_id AS BIGINT\n        ) AS stay_id,\n\n        TRIM(\n            CAST(\n                p.drug AS VARCHAR\n            )\n        ) AS drug,\n\n        regexp_replace(\n            TRIM(\n                CAST(\n                    p.ndc AS VARCHAR\n                )\n            ),\n            '[^0-9]',\n            '',\n            'g'\n        ) AS ndc\n\n    FROM read_csv_auto(\n        '{prescriptions_path}',\n        header=true,\n        all_varchar=true,\n        union_by_name=true\n    ) AS p\n\n    INNER JOIN read_csv_auto(\n        '{icustays_path}',\n        header=true,\n        all_varchar=true,\n        union_by_name=true\n    ) AS i\n\n        ON TRY_CAST(\n            p.subject_id AS BIGINT\n        ) = TRY_CAST(\n            i.subject_id AS BIGINT\n        )\n\n        AND TRY_CAST(\n            p.hadm_id AS BIGINT\n        ) = TRY_CAST(\n            i.hadm_id AS BIGINT\n        )\n    ")
con.execute("\n    CREATE OR REPLACE TEMP TABLE\n    ndc_stay_atc3 AS\n\n    SELECT DISTINCT\n\n        b.subject_id,\n        b.stay_id,\n        b.drug,\n        m.ATC3,\n        'ndc_fallback' AS mapping_source\n\n    FROM medication_baseline AS b\n\n    INNER JOIN unresolved_drugs_df AS u\n\n        ON b.drug = u.drug\n\n    INNER JOIN raw_stay_ndcs AS r\n\n        ON b.subject_id = r.subject_id\n\n        AND b.stay_id = r.stay_id\n\n        AND b.drug = r.drug\n\n    INNER JOIN successful_ndc_atc3 AS m\n\n        ON r.ndc = m.ndc\n    ")
con.execute('\n    CREATE OR REPLACE TEMP TABLE\n    combined_drug_atc3 AS\n\n    SELECT\n        subject_id,\n        stay_id,\n        drug,\n        ATC3,\n        mapping_source\n\n    FROM text_stay_atc3\n\n    UNION ALL\n\n    SELECT\n        subject_id,\n        stay_id,\n        drug,\n        ATC3,\n        mapping_source\n\n    FROM ndc_stay_atc3\n    ')
con.execute('\n    CREATE OR REPLACE TEMP TABLE\n    final_stay_atc3 AS\n\n    SELECT DISTINCT\n\n        subject_id,\n        stay_id,\n        ATC3\n\n    FROM combined_drug_atc3\n\n    WHERE ATC3 IS NOT NULL\n    ')
con.execute(f"\n    COPY final_stay_atc3\n    TO '{output_long_path}'\n    (\n        FORMAT PARQUET,\n        COMPRESSION ZSTD\n    )\n    ")
audit = con.execute('\n    WITH\n\n    baseline AS (\n\n        SELECT\n            COUNT(*) AS total_presences\n\n        FROM medication_baseline\n\n    ),\n\n    text_covered AS (\n\n        SELECT COUNT(*) AS n\n\n        FROM (\n\n            SELECT DISTINCT\n                subject_id,\n                stay_id,\n                drug\n\n            FROM text_stay_atc3\n        )\n    ),\n\n    ndc_covered AS (\n\n        SELECT COUNT(*) AS n\n\n        FROM (\n\n            SELECT DISTINCT\n                subject_id,\n                stay_id,\n                drug\n\n            FROM ndc_stay_atc3\n        )\n    ),\n\n    all_covered AS (\n\n        SELECT COUNT(*) AS n\n\n        FROM (\n\n            SELECT DISTINCT\n                subject_id,\n                stay_id,\n                drug\n\n            FROM combined_drug_atc3\n        )\n    )\n\n    SELECT\n\n        baseline.total_presences\n            AS total_stay_drug_presences,\n\n        text_covered.n\n            AS text_covered_presences,\n\n        ndc_covered.n\n            AS ndc_fallback_presences,\n\n        all_covered.n\n            AS total_covered_presences,\n\n        baseline.total_presences\n            - all_covered.n\n            AS uncovered_presences,\n\n        CAST(\n            all_covered.n AS DOUBLE\n        )\n        /\n        baseline.total_presences\n            AS exact_atc3_coverage\n\n    FROM baseline,\n         text_covered,\n         ndc_covered,\n         all_covered\n    ').fetchdf()
feature_audit = con.execute('\n    SELECT\n\n        COUNT(\n            DISTINCT ATC3\n        ) AS n_atc3_categories,\n\n        COUNT(\n            DISTINCT stay_id\n        ) AS stays_with_atc3,\n\n        COUNT(*) AS stay_atc3_rows\n\n    FROM final_stay_atc3\n    ').fetchdf()
con.close()
audit.to_csv(FINAL_ATC3_AUDIT_CSV, index=False)
print()
print('======================================')
print('FINAL EXACT ATC3 REPRESENTATION')
print('======================================')
print('Total baseline stay-drug presences:', int(audit.loc[0, 'total_stay_drug_presences']))
print('Covered by text mapping:', int(audit.loc[0, 'text_covered_presences']))
print('Recovered by NDC fallback:', int(audit.loc[0, 'ndc_fallback_presences']))
print('Total ATC3-covered presences:', int(audit.loc[0, 'total_covered_presences']))
print('Remaining uncovered presences:', int(audit.loc[0, 'uncovered_presences']))
print('EXACT overall ATC3 coverage:', f"{audit.loc[0, 'exact_atc3_coverage']:.2%}")
print()
print('Final distinct ATC3 categories:', int(feature_audit.loc[0, 'n_atc3_categories']))
print('Stays with >=1 ATC3 feature:', int(feature_audit.loc[0, 'stays_with_atc3']))
print('Final stay-ATC3 rows:', int(feature_audit.loc[0, 'stay_atc3_rows']))
print()
print('Saved final ATC3 long representation:', FINAL_ATC3_LONG_PARQUET)
print('Saved exact coverage audit:', FINAL_ATC3_AUDIT_CSV)


In [ ]:
ATC3_WIDE_PARQUET = os.path.join(REDUCTION_DIR, 'final_medication_atc3_binary_features.parquet')
atc3_long_path = FINAL_ATC3_LONG_PARQUET.replace('\\', '/')
wide_output_path = ATC3_WIDE_PARQUET.replace('\\', '/')
con = duckdb.connect()
con.execute("PRAGMA memory_limit='12GB'")
if os.path.exists(NOIMP_PARQUET):
    cohort_path = NOIMP_PARQUET.replace('\\', '/')
    con.execute(f"\n        CREATE OR REPLACE TEMP TABLE final_cohort AS\n\n        SELECT DISTINCT\n\n            CAST(\n                subject_id AS BIGINT\n            ) AS subject_id,\n\n            CAST(\n                stay_id AS BIGINT\n            ) AS stay_id\n\n        FROM read_parquet(\n            '{cohort_path}'\n        )\n        ")
else:
    cohort_path = NOIMP_CSV.replace('\\', '/')
    con.execute(f"\n        CREATE OR REPLACE TEMP TABLE final_cohort AS\n\n        SELECT DISTINCT\n\n            TRY_CAST(\n                subject_id AS BIGINT\n            ) AS subject_id,\n\n            TRY_CAST(\n                stay_id AS BIGINT\n            ) AS stay_id\n\n        FROM read_csv_auto(\n            '{cohort_path}',\n            header=true\n        )\n        ")
con.execute(f"\n    CREATE OR REPLACE TEMP VIEW atc3_long AS\n\n    SELECT DISTINCT\n\n        CAST(\n            subject_id AS BIGINT\n        ) AS subject_id,\n\n        CAST(\n            stay_id AS BIGINT\n        ) AS stay_id,\n\n        CAST(\n            ATC3 AS VARCHAR\n        ) AS ATC3\n\n    FROM read_parquet(\n        '{atc3_long_path}'\n    )\n\n    WHERE ATC3 IS NOT NULL\n    ")
con.execute('\n    CREATE OR REPLACE TEMP TABLE atc3_pivot AS\n\n    PIVOT (\n\n        SELECT\n\n            subject_id,\n            stay_id,\n            ATC3,\n            1 AS present\n\n        FROM atc3_long\n    )\n\n    ON ATC3\n\n    USING MAX(present)\n\n    GROUP BY\n        subject_id,\n        stay_id\n    ')
atc3_wide = con.execute('\n    SELECT *\n    FROM atc3_pivot\n    ORDER BY subject_id, stay_id\n    ').fetchdf()
raw_atc_columns = [c for c in atc3_wide.columns if c not in {'subject_id', 'stay_id'}]
print('ATC3 columns produced by pivot:', len(raw_atc_columns))
rename_map = {c: f'med_atc3_{c}' for c in raw_atc_columns}
atc3_wide = atc3_wide.rename(columns=rename_map)
med_feature_columns = [c for c in atc3_wide.columns if c.startswith('med_atc3_')]
cohort_df = con.execute('\n    SELECT\n        subject_id,\n        stay_id\n\n    FROM final_cohort\n\n    ORDER BY\n        subject_id,\n        stay_id\n    ').fetchdf()
con.close()
atc3_wide = cohort_df.merge(atc3_wide, on=['subject_id', 'stay_id'], how='left', validate='one_to_one')
atc3_wide[med_feature_columns] = atc3_wide[med_feature_columns].fillna(0).astype(np.uint8)
assert len(atc3_wide) == 74501, 'Unexpected cohort size.'
assert len(med_feature_columns) == 192, 'Expected exactly 192 ATC3 medication features.'
unique_values = set(np.unique(atc3_wide[med_feature_columns].to_numpy()))
assert unique_values.issubset({0, 1}), f'Non-binary values found: {unique_values}'
row_sums = atc3_wide[med_feature_columns].sum(axis=1)
zero_atc3_stays = int((row_sums == 0).sum())
atc3_wide.to_parquet(ATC3_WIDE_PARQUET, index=False)
print()
print('======================================')
print('FINAL BINARY ATC3 MEDICATION MATRIX')
print('======================================')
print('Matrix shape:', atc3_wide.shape)
print('Final cohort stays:', len(atc3_wide))
print('ATC3 medication features:', len(med_feature_columns))
print('Binary values present:', sorted(unique_values))
print('Stays with >=1 ATC3 feature:', int((row_sums > 0).sum()))
print('Stays with zero ATC3 features:', zero_atc3_stays)
print('Total positive stay-ATC3 cells:', int(row_sums.sum()))
possible_cells = len(atc3_wide) * len(med_feature_columns)
density = row_sums.sum() / possible_cells
print('ATC3 feature density:', f'{density:.6%}')
print('ATC3 feature sparsity:', f'{1 - density:.6%}')
print()
print('Saved binary ATC3 matrix:', ATC3_WIDE_PARQUET)


## 8. Construct the reduced A1 dataset

The 5,222 original medication indicators are removed and the 192 ATC3 indicators are merged in. Eighteen medication names collided with pre-existing non-medication columns during the original Pandas merge; the medication `_y` columns are removed while the corresponding non-medication `_x` columns are retained. No rows are removed.


In [ ]:
REDUCED_NOIMP_PARQUET = os.path.join(REDUCTION_DIR, 'icu_data_no_imputation_atc3_reduced.parquet')
REDUCED_NOIMP_CSV = os.path.join(REDUCTION_DIR, 'icu_data_no_imputation_atc3_reduced.csv')
if os.path.exists(NOIMP_PARQUET):
    original_icu = pd.read_parquet(NOIMP_PARQUET)
else:
    original_icu = pd.read_csv(NOIMP_CSV, low_memory=True)
print('Original dataset shape:', original_icu.shape)
pharm_path = PROCESSED_PHARM.replace('\\', '/')
con = duckdb.connect()
raw_drug_names = con.execute(f"\n        SELECT DISTINCT\n\n            CAST(\n                drug AS VARCHAR\n            ) AS drug\n\n        FROM read_parquet(\n            '{pharm_path}'\n        )\n\n        WHERE drug IS NOT NULL\n        ").fetchdf()['drug'].tolist()
con.close()
print('Raw medication names in pharmacy checkpoint:', len(raw_drug_names))
assert len(raw_drug_names) == 5222
original_column_set = set(original_icu.columns)
exact_med_columns = [drug for drug in raw_drug_names if drug in original_column_set]
missing_exact_names = [drug for drug in raw_drug_names if drug not in original_column_set]
print('Exact-name medication columns:', len(exact_med_columns))
print('Collision medication names:', len(missing_exact_names))
collision_med_columns = [f'{drug}_y' for drug in missing_exact_names]
collision_nonmed_columns = [f'{drug}_x' for drug in missing_exact_names]
assert all((col in original_column_set for col in collision_med_columns)), 'At least one expected *_y medication column is missing.'
assert all((col in original_column_set for col in collision_nonmed_columns)), 'At least one expected *_x non-medication column is missing.'
print('Collision *_y medication columns:', len(collision_med_columns))
print('Collision *_x columns retained:', len(collision_nonmed_columns))
original_medication_columns = exact_med_columns + collision_med_columns
original_medication_columns = list(dict.fromkeys(original_medication_columns))
print()
print('Total original medication features identified:', len(original_medication_columns))
assert len(original_medication_columns) == 5222, 'Expected exactly 5,222 original medication features.'
non_medication_data = original_icu.drop(columns=original_medication_columns).copy()
print('Columns after removing raw medications:', non_medication_data.shape[1])
assert non_medication_data.shape[1] == original_icu.shape[1] - 5222
missing_retained_x = [col for col in collision_nonmed_columns if col not in non_medication_data.columns]
assert len(missing_retained_x) == 0, 'A non-medication *_x collision column was accidentally removed.'
print('All collision *_x columns retained:', len(collision_nonmed_columns))
atc3_features = pd.read_parquet(ATC3_WIDE_PARQUET)
atc3_med_columns = [col for col in atc3_features.columns if col.startswith('med_atc3_')]
print('ATC3 medication features to add:', len(atc3_med_columns))
assert len(atc3_med_columns) == 192
assert len(atc3_features) == 74501
reduced_icu = non_medication_data.merge(atc3_features, on=['subject_id', 'stay_id'], how='left', validate='one_to_one')
missing_atc_cells = int(reduced_icu[atc3_med_columns].isna().sum().sum())
print('Missing ATC3 cells after merge:', missing_atc_cells)
assert missing_atc_cells == 0
reduced_icu[atc3_med_columns] = reduced_icu[atc3_med_columns].astype(np.uint8)
assert len(reduced_icu) == 74501
assert reduced_icu['stay_id'].nunique() == 74501
expected_reduced_columns = original_icu.shape[1] - 5222 + 192
assert reduced_icu.shape[1] == expected_reduced_columns
old_med_still_present = [col for col in original_medication_columns if col in reduced_icu.columns]
assert len(old_med_still_present) == 0
retained_collision_x = [col for col in collision_nonmed_columns if col in reduced_icu.columns]
assert len(retained_collision_x) == 18
required_columns = ['subject_id', 'stay_id', 'los', 'age', 'gender', 'insurance', 'race']
for col in required_columns:
    assert col in reduced_icu.columns, f'Required column missing: {col}'
reduced_icu.to_parquet(REDUCED_NOIMP_PARQUET, index=False)
reduced_icu.to_csv(REDUCED_NOIMP_CSV, index=False)
print()
print('======================================')
print('A1 REDUCED-FEATURE DATASET')
print('======================================')
print('Original dataset shape:', original_icu.shape)
print('Exact-name medication features removed:', len(exact_med_columns))
print('Collision *_y medication features removed:', len(collision_med_columns))
print('Total original medication features removed:', len(original_medication_columns))
print('Collision *_x non-medication features retained:', len(retained_collision_x))
print('ATC3 medication features added:', len(atc3_med_columns))
print('Medication dimensionality:', f'{len(original_medication_columns)} -> {len(atc3_med_columns)}')
print('Final reduced dataset shape:', reduced_icu.shape)
print('Rows retained:', len(reduced_icu))
print('Unique ICU stays:', reduced_icu['stay_id'].nunique())
print('Unique patients:', reduced_icu['subject_id'].nunique())
print()
print('Saved reduced Parquet:', REDUCED_NOIMP_PARQUET)
print('Saved reduced CSV:', REDUCED_NOIMP_CSV)
del original_icu
del non_medication_data
del atc3_features
gc.collect()


## 9. Reproduce the original patient-level partitions

A1 uses the same literal LOS rule (`los > 4`), outer 80/20 patient split, and inner 80/20 patient split as A0, with `random_state=42`. The cell also verifies that all four patient partitions are identical to the baseline and that there is no patient leakage.


In [ ]:
from sklearn.model_selection import train_test_split
icu_data = pd.read_csv(REDUCED_NOIMP_CSV, low_memory=True)
print('Reduced CSV shape before los_type:', icu_data.shape)
assert icu_data.shape == (74501, 704)
condition = (icu_data['los'] > 4).map({True: 'Long', False: 'Short'})
icu_data['los_type'] = condition
print()
print('LOS class counts:')
print(icu_data['los_type'].value_counts())
patient_ids = icu_data['subject_id'].unique()
train_patient_ids, validation_patient_ids = train_test_split(patient_ids, test_size=0.2, random_state=42)
train_data = icu_data[icu_data['subject_id'].isin(train_patient_ids)].copy()
validation_data = icu_data[icu_data['subject_id'].isin(validation_patient_ids)].copy()
patient_ids_train = train_data['subject_id'].unique()
train_train_ids, test_train_ids = train_test_split(patient_ids_train, test_size=0.2, random_state=42)
train_train_data = train_data[train_data['subject_id'].isin(train_train_ids)].copy()
test_train_data = train_data[train_data['subject_id'].isin(test_train_ids)].copy()
original_subjects = pd.read_csv(NOIMP_CSV, usecols=['subject_id'], low_memory=True)
original_patient_ids = original_subjects['subject_id'].unique()
orig_train_ids, orig_validation_ids = train_test_split(original_patient_ids, test_size=0.2, random_state=42)
original_train_subject_sequence = original_subjects.loc[original_subjects['subject_id'].isin(orig_train_ids), 'subject_id'].unique()
orig_train_train_ids, orig_test_train_ids = train_test_split(original_train_subject_sequence, test_size=0.2, random_state=42)
outer_train_same = set(train_patient_ids) == set(orig_train_ids)
outer_validation_same = set(validation_patient_ids) == set(orig_validation_ids)
inner_train_same = set(train_train_ids) == set(orig_train_train_ids)
inner_test_same = set(test_train_ids) == set(orig_test_train_ids)
assert outer_train_same
assert outer_validation_same
assert inner_train_same
assert inner_test_same
assert set(train_patient_ids).isdisjoint(set(validation_patient_ids))
assert set(train_train_ids).isdisjoint(set(test_train_ids))
print()
print('======================================')
print('A1 PATIENT-SPLIT INTEGRITY')
print('======================================')
print('Total rows:', len(icu_data))
print('Total patients:', icu_data['subject_id'].nunique())
print()
print('Outer train rows:', len(train_data))
print('Outer validation rows:', len(validation_data))
print()
print('Inner training rows:', len(train_train_data))
print('Inner testing rows:', len(test_train_data))
print()
print('Outer train patients:', len(train_patient_ids))
print('Outer validation patients:', len(validation_patient_ids))
print('Inner train patients:', len(train_train_ids))
print('Inner test patients:', len(test_train_ids))
print()
print('Outer train identical to baseline:', outer_train_same)
print('Outer validation identical to baseline:', outer_validation_same)
print('Inner train identical to baseline:', inner_train_same)
print('Inner test identical to baseline:', inner_test_same)
print()
print('Train/validation patient leakage:', len(set(train_patient_ids) & set(validation_patient_ids)))
print('Inner train/test patient leakage:', len(set(train_train_ids) & set(test_train_ids)))
del original_subjects


## 10. A1 XGBoost evaluation

The original exclusion list, numeric dtype selection, `StandardScaler`, `XGBClassifier(objective='binary:logistic', random_state=42)`, and 0.5 decision threshold are retained. The internal test set is evaluated first, followed by the untouched outer held-out set using the same fitted scaler and model.


In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, confusion_matrix, classification_report, accuracy_score
exclude_columns = ['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los', 'los_type', 'age', 'gender', 'insurance', 'race', 'PHYSICIAN REFERRAL']
feature_columns = [col for col in train_data.columns if col not in exclude_columns]
numeric_columns = train_train_data[feature_columns].select_dtypes(include=['float64', 'int64']).columns
model_atc3_columns = [col for col in numeric_columns if col.startswith('med_atc3_')]
print('======================================')
print('A1 MODEL FEATURE AUDIT')
print('======================================')
print('Candidate feature columns:', len(feature_columns))
print('Numeric model features:', len(numeric_columns))
print('ATC3 medication features selected:', len(model_atc3_columns))
assert len(model_atc3_columns) == 192, 'Not all 192 ATC3 features entered the model.'
for sensitive_col in ['race', 'insurance', 'gender']:
    assert sensitive_col not in numeric_columns
X_train = train_train_data[numeric_columns]
y_train = train_train_data['los_type']
X_test = test_train_data[numeric_columns]
y_test = test_train_data['los_type']
y_train = y_train.map({'Short': 0, 'Long': 1})
y_test = y_test.map({'Short': 0, 'Long': 1})
print()
print('Internal training rows:', len(X_train))
print('Internal testing rows:', len(X_test))
print('Training class counts:')
print(y_train.value_counts().sort_index())
print('Testing class counts:')
print(y_test.value_counts().sort_index())
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
a1_model = XGBClassifier(objective='binary:logistic', random_state=42)
a1_model.fit(X_train_scaled, y_train)
y_pred_prob = a1_model.predict_proba(X_test_scaled)[:, 1]
threshold = 0.5
y_pred = (y_pred_prob >= threshold).astype(int)
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)
precision, recall, _ = precision_recall_curve(y_test, y_pred_prob)
average_precision = average_precision_score(y_test, y_pred_prob)
cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
cr = classification_report(y_test, y_pred, digits=2)
print()
print('======================================')
print('A1 REDUCED-FEATURE XGBOOST')
print('INTERNAL TEST RESULTS')
print('======================================')
print(f'AUC-ROC: {roc_auc:.4f}')
print(f'AUC-PR:  {average_precision:.4f}')
print(f'Accuracy: {accuracy:.4f}')
print()
print('Confusion Matrix:')
print(cm)
print()
print('Classification Report:')
print(cr)


In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, confusion_matrix, classification_report, accuracy_score
X_verify = validation_data[numeric_columns]
y_verify = validation_data['los_type'].map({'Short': 0, 'Long': 1})
print('======================================')
print('A1 OUTER HELD-OUT FEATURE AUDIT')
print('======================================')
print('Held-out rows:', len(X_verify))
print('Model features:', len(numeric_columns))
heldout_atc3_columns = [col for col in numeric_columns if col.startswith('med_atc3_')]
print('ATC3 medication features:', len(heldout_atc3_columns))
print('Held-out class counts:')
print(y_verify.value_counts().sort_index())
assert len(X_verify) == 14986
assert len(heldout_atc3_columns) == 192
X_verify_scaled = scaler.transform(X_verify)
y_verify_pred_prob = a1_model.predict_proba(X_verify_scaled)[:, 1]
threshold = 0.5
y_verify_pred = (y_verify_pred_prob > threshold).astype(int)
fpr_verify, tpr_verify, _ = roc_curve(y_verify, y_verify_pred_prob)
roc_auc_verify = auc(fpr_verify, tpr_verify)
precision_verify, recall_verify, _ = precision_recall_curve(y_verify, y_verify_pred_prob)
average_precision_verify = average_precision_score(y_verify, y_verify_pred_prob)
cm_verify = confusion_matrix(y_verify, y_verify_pred)
accuracy_verify = accuracy_score(y_verify, y_verify_pred)
cr_verify = classification_report(y_verify, y_verify_pred, digits=2)
print()
print('======================================')
print('A1 REDUCED-FEATURE XGBOOST')
print('OUTER HELD-OUT RESULTS')
print('======================================')
print(f'AUC-ROC: {roc_auc_verify:.4f}')
print(f'AUC-PR:  {average_precision_verify:.4f}')
print(f'Accuracy: {accuracy_verify:.4f}')
print()
print('Confusion Matrix:')
print(cm_verify)
print()
print('Classification Report:')
print(cr_verify)


## 11. Held-out subgroup evaluation

Race and insurance are not predictors. The same 14,986 held-out predictions are stratified by subgroup to compute confusion-matrix counts, accuracy, Long-stay precision/recall/F1, TPR, FPR, and positive prediction rate. Small subgroups should be interpreted cautiously.


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def subgroup_summary(df, group_column):
    rows = []
    for group in df[group_column].dropna().unique():
        subset = df.loc[df[group_column] == group]
        yt, yp = subset["Actual"].astype(int), subset["Predicted"].astype(int)
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        rows.append({
            group_column: group, "n": len(subset), "TN": tn, "FP": fp, "FN": fn, "TP": tp,
            "accuracy": accuracy_score(yt, yp),
            "long_precision": precision_score(yt, yp, zero_division=0),
            "long_recall": recall_score(yt, yp, zero_division=0),
            "long_f1": f1_score(yt, yp, zero_division=0),
            "TPR": tp / (tp + fn) if tp + fn else np.nan,
            "FPR": fp / (fp + tn) if fp + tn else np.nan,
            "positive_prediction_rate": yp.mean(),
        })
    return pd.DataFrame(rows)

fairness_results = pd.DataFrame({
    "Actual": y_verify.to_numpy(),
    "Predicted": np.asarray(y_verify_pred),
    "Insurance": validation_data["insurance"].to_numpy(),
    "Race": validation_data["race"].to_numpy(),
})
assert len(fairness_results) == 14986
insurance_fairness = subgroup_summary(fairness_results, "Insurance")
race_fairness = subgroup_summary(fairness_results, "Race")

print("Missing insurance:", int(fairness_results['Insurance'].isna().sum()))
print("Missing race:", int(fairness_results['Race'].isna().sum()))
display(insurance_fairness.sort_values("n", ascending=False))
display(race_fairness.sort_values("n", ascending=False))


## 12. Direct A0 versus A1 comparison

The full 5,222-medication A0 model is recreated using the same patient IDs and modeling pipeline. This provides a controlled comparison in which medication representation is the principal experimental change.


In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_curve, auc, average_precision_score, accuracy_score, precision_score, recall_score, f1_score
baseline_icu = pd.read_csv(NOIMP_CSV, low_memory=True)
baseline_icu['los_type'] = (baseline_icu['los'] > 4).map({True: 'Long', False: 'Short'})
print('Original baseline shape:', baseline_icu.shape)
baseline_train = baseline_icu[baseline_icu['subject_id'].isin(train_patient_ids)].copy()
baseline_validation = baseline_icu[baseline_icu['subject_id'].isin(validation_patient_ids)].copy()
baseline_inner_train = baseline_train[baseline_train['subject_id'].isin(train_train_ids)].copy()
baseline_inner_test = baseline_train[baseline_train['subject_id'].isin(test_train_ids)].copy()
assert len(baseline_inner_train) == 47543
assert len(baseline_inner_test) == 11972
assert len(baseline_validation) == 14986
exclude_columns = ['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los', 'los_type', 'age', 'gender', 'insurance', 'race', 'PHYSICIAN REFERRAL']
baseline_feature_columns = [col for col in baseline_train.columns if col not in exclude_columns]
baseline_numeric_columns = baseline_inner_train[baseline_feature_columns].select_dtypes(include=['float64', 'int64']).columns
print('Baseline model features:', len(baseline_numeric_columns))
Xb_train = baseline_inner_train[baseline_numeric_columns]
yb_train = baseline_inner_train['los_type'].map({'Short': 0, 'Long': 1})
baseline_scaler = StandardScaler()
Xb_train_scaled = baseline_scaler.fit_transform(Xb_train)
baseline_model = XGBClassifier(objective='binary:logistic', random_state=42)
baseline_model.fit(Xb_train_scaled, yb_train)
Xb_verify = baseline_validation[baseline_numeric_columns]
yb_verify = baseline_validation['los_type'].map({'Short': 0, 'Long': 1})
Xb_verify_scaled = baseline_scaler.transform(Xb_verify)
baseline_pred_prob = baseline_model.predict_proba(Xb_verify_scaled)[:, 1]
baseline_pred = (baseline_pred_prob > 0.5).astype(int)
baseline_cm = confusion_matrix(yb_verify, baseline_pred)
baseline_fpr_curve, baseline_tpr_curve, _ = roc_curve(yb_verify, baseline_pred_prob)
baseline_auc = auc(baseline_fpr_curve, baseline_tpr_curve)
baseline_pr = average_precision_score(yb_verify, baseline_pred_prob)
baseline_acc = accuracy_score(yb_verify, baseline_pred)
print()
print('======================================')
print('ORIGINAL BASELINE HELD-OUT CHECK')
print('======================================')
print(f'AUC-ROC: {baseline_auc:.4f}')
print(f'AUC-PR:  {baseline_pr:.4f}')
print(f'Accuracy: {baseline_acc:.4f}')
print('Confusion Matrix:')
print(baseline_cm)
baseline_fairness = pd.DataFrame({'Actual': yb_verify.to_numpy(), 'Predicted': baseline_pred, 'Insurance': baseline_validation['insurance'].to_numpy(), 'Race': baseline_validation['race'].to_numpy()})

def compact_group_summary(df, group_column):
    rows = []
    for group in df[group_column].dropna().unique():
        subset = df[df[group_column] == group]
        yt = subset['Actual'].astype(int)
        yp = subset['Predicted'].astype(int)
        cm = confusion_matrix(yt, yp, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        rows.append({group_column: group, 'n': len(subset), 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp, 'accuracy': accuracy_score(yt, yp), 'long_precision': precision_score(yt, yp, zero_division=0), 'long_recall': recall_score(yt, yp, zero_division=0), 'long_f1': f1_score(yt, yp, zero_division=0), 'TPR': tp / (tp + fn) if tp + fn > 0 else np.nan, 'FPR': fp / (fp + tn) if fp + tn > 0 else np.nan, 'positive_prediction_rate': yp.mean()})
    return pd.DataFrame(rows)
baseline_insurance_fairness = compact_group_summary(baseline_fairness, 'Insurance')
baseline_race_fairness = compact_group_summary(baseline_fairness, 'Race')
insurance_comparison = baseline_insurance_fairness.merge(insurance_fairness, on='Insurance', suffixes=('_A0', '_A1'))
race_comparison = baseline_race_fairness.merge(race_fairness, on='Race', suffixes=('_A0', '_A1'))
for df in [insurance_comparison, race_comparison]:
    df['delta_long_recall'] = df['long_recall_A1'] - df['long_recall_A0']
    df['delta_FPR'] = df['FPR_A1'] - df['FPR_A0']
    df['delta_long_f1'] = df['long_f1_A1'] - df['long_f1_A0']
print()
print('======================================')
print('A0 vs A1 INSURANCE FAIRNESS')
print('======================================')
display(insurance_comparison[['Insurance', 'n_A0', 'long_recall_A0', 'long_recall_A1', 'delta_long_recall', 'FPR_A0', 'FPR_A1', 'delta_FPR', 'long_f1_A0', 'long_f1_A1', 'delta_long_f1']].sort_values('n_A0', ascending=False))
print()
print('======================================')
print('A0 vs A1 RACE FAIRNESS')
print('======================================')
display(race_comparison[['Race', 'n_A0', 'long_recall_A0', 'long_recall_A1', 'delta_long_recall', 'FPR_A0', 'FPR_A1', 'delta_FPR', 'long_f1_A0', 'long_f1_A1', 'delta_long_f1']].sort_values('n_A0', ascending=False))


## 13. Results from the completed run

### 13.1 Mapping and dimensionality

| Quantity | Completed-run result |
|---|---:|
| Final cohort ICU stays | 74,501 |
| Raw medication columns in pharmacy checkpoint | 5,222 |
| Distinct medication names active in final cohort | 5,097 |
| Raw medication sparsity | 99.09% |
| Drug names resolved to ≥1 RxNorm ingredient | 2,560 |
| Ingredient exposure-weighted coverage | 65.47% |
| Drug names with text-derived ATC3 | 2,359 |
| Text-derived ATC3 exposure coverage | 62.53% |
| Unresolved names with ≥1 candidate NDC | 1,693 |
| Candidate NDCs with active RxNorm product concept | 1,846 |
| Unresolved drug names recovered with NDC-derived ATC3 | 1,283 |
| Final distinct ATC3 categories | **192** |
| Final ATC3 stay-drug exposure coverage | **78.88%** |
| Medication feature reduction | **5,222 → 192 (96.3%)** |
| Total A1 model predictors | **691** |

Of 3,460,579 baseline stay-drug presences, 2,163,886 were covered by text mapping and 565,806 additional presences were recovered through the NDC fallback. One ICU stay had no mapped ATC3 category and was retained with an all-zero medication vector.

### 13.2 Predictive performance

| Metric | A0: raw medications | A1: ATC3 | Difference |
|---|---:|---:|---:|
| Internal ROC-AUC | ~0.88 | 0.8681 | modest decrease |
| Internal PR-AUC | ~0.79 | 0.7751 | modest decrease |
| Internal accuracy | ~0.83 | 0.8234 | modest decrease |
| Held-out ROC-AUC | 0.8791 | 0.8700 | -0.0091 |
| Held-out PR-AUC | 0.7858 | 0.7737 | -0.0121 |
| Held-out accuracy | 0.8312 | 0.8225 | -0.0087 |
| Held-out false positives | 893 | 896 | +3 |
| Held-out false negatives | 1,636 | 1,764 | +128 |

The principal utility cost was reduced sensitivity for Long stays; the false-positive count was essentially unchanged.

### 13.3 Insurance-group comparison

| Insurance | Long recall A0 | Long recall A1 | Δ recall | FPR A0 | FPR A1 | Δ FPR |
|---|---:|---:|---:|---:|---:|---:|
| Medicare | 0.6225 | 0.5923 | -0.0302 | 0.0890 | 0.0892 | +0.0002 |
| Private | 0.6244 | 0.5926 | -0.0319 | 0.0687 | 0.0698 | +0.0011 |
| Medicaid | 0.6662 | 0.6531 | -0.0131 | 0.0980 | 0.0986 | +0.0006 |
| Other | 0.6429 | 0.5982 | -0.0446 | 0.0791 | 0.0751 | -0.0040 |

The `No charge` category contained only five observations and should not be interpreted substantively.

### 13.4 Race-group comparison

| Race | Long recall A0 | Long recall A1 | Δ recall | FPR A0 | FPR A1 | Δ FPR |
|---|---:|---:|---:|---:|---:|---:|
| White | 0.6161 | 0.5816 | -0.0345 | 0.0807 | 0.0824 | +0.0017 |
| Other | 0.7024 | 0.6873 | -0.0151 | 0.1012 | 0.1059 | +0.0048 |
| Black/African American | 0.6087 | 0.5818 | -0.0269 | 0.0859 | 0.0784 | -0.0075 |
| Hispanic/Latino | 0.6229 | 0.5886 | -0.0343 | 0.0879 | 0.0641 | -0.0238 |
| Asian | 0.5929 | 0.5929 | 0.0000 | 0.0939 | 0.0939 | 0.0000 |

Pacific Islander (`n=48`) and Multiple Race/Ethnicity (`n=15`) estimates are retained by the code but should be treated as unstable because of their small sample sizes.


## 14. Generated analysis artifacts

The workflow writes the principal intermediate and final artifacts to `data/medication_feature_reduction`:

- `drug_direct_atc_mapping_long.csv` — primary text-based RxNorm/ATC mapping checkpoint.
- `pin_min_to_in_atc_recovery.csv` — PIN/MIN-to-IN recovery records.
- `drug_direct_atc_mapping_long_recovered.csv` — combined text mapping after PIN/MIN recovery.
- `unresolved_drug_ndc_availability.csv` — NDC availability audit for text-unresolved medication names.
- `ndc_rxnorm_status_full.csv` — RxNorm status for candidate NDCs.
- `ndc_active_product_to_atc_full.csv` — active NDC product-to-ingredient-to-ATC recovery.
- `final_medication_atc3_stay_long.parquet` — final stay-level ATC3 long representation.
- `final_medication_atc3_binary_features.parquet` — binary 192-feature ATC3 matrix.
- `icu_data_no_imputation_atc3_reduced.parquet` and `.csv` — final A1 reduced-feature dataset.


## 15. Methodological scope and limitations

1. **Controlled intervention:** A1 changes the medication representation but not the LOS target, cohort, patient partitions, non-medication feature construction, XGBoost configuration, or held-out evaluation.
2. **ATC3 selection:** ATC3 was chosen before model evaluation as a clinically interpretable compromise between broad grouping and high-dimensional drug-level encoding; it was not selected using held-out performance.
3. **Multiple assignments:** Multiple valid ATC assignments are retained rather than forcing a drug into one category.
4. **Incomplete mapping:** Approximately 21.12% of stay-drug presences remain without an ATC3 assignment; absence of an ATC3 indicator can therefore reflect either true absence or unmapped medication information.
5. **NDC linkage:** NDC recovery relies on admission-level reconstruction because raw prescriptions do not contain ICU `stay_id`. This limitation should be reported when the feature-reduction method is described.
